In [1]:
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import geopandas as gpd
import osmnx as ox
from typing import List

# Import necessary libraries
import pandas as pd
from collections import defaultdict
import geopandas as gpd
from shapely import wkt
import shapely

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pickle
%matplotlib inline
import datetime
import time
import numpy as np
import xml.etree.ElementTree as ET 
import matsim

import utm
from shapely.geometry import Polygon, Point
import gzip
from matplotlib import cm

import matplotlib.ticker as ticker
import matplotlib.font_manager as font_manager
import matplotlib as mpl
from matplotlib.lines import Line2D
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap
import json



In [1]:
# =========================================================================================
# NOTEBOOK NOTE:
# -----------------------------------------------------------------------------------------
# This file is a quick and dirty workbench to get results out of the batch runs.
# It is messy on purpose. Stability and correctness first, cleanup later.
# Sorry to anyone reading this :DD Quick and dirty for evaluation. No Time, Paper Deadline...
# It’s not clean, not modular, and definitely not pretty.
# The only goal right now: make the plots look right and get the paper submitted.
# =============================================================================
# NOTE
# -----------------------------------------------------------------------------
#
# What this does
# - Discovers MATSim runs in BATCH_DIR
# - Parses events and carriers
# - Builds per vehicle stats and EV assignments
# - Computes emissions (drive, idle, cold) and aggregates to 15 min bins
# - Writes a bunch of .pkl bundles next to the runs
#
# Known mess
# - Hard coded paths
# - Globals all over the place
# - Mixed responsibilities inside run_batch
# - Missing guards for some None cases
#
# TODO after the deadline
# [ ] Move helpers into utils modules
# [ ] Replace prints with structured logging
# [ ] Add type hints and docstrings
# [ ] Centralize config and paths
# [ ] Unit tests for parsing and clipping
# [ ] Package this into a clean CLI
# =============================================================================

In [2]:
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)

def determine_area_type(raumtyp):
    if raumtyp in [1, 2, 3]:
        return 'Urban'
    elif raumtyp in [4, 5, 6]:
        return 'Suburban'
    elif raumtyp in [7, 8]:
        return 'Rural'
    else:
        return 'unknown'
    
category_dict = {
    "1": "Metropolitan Center",
    "2": "High-Density Residential Use",
    "3": "Dense Mixed Use",
    "4": "Residential Use",
    "5": "Industrial Use",
    "6": "Urbanized Periphery",
    "7": "Rural with Industrial Influence",
    "8": "Rural without Industrial Influence"
}


# Wende die Funktion auf die Spalte raumtyp an und erstelle die neue Spalte area_type
regionclusters['area_type_agg'] = regionclusters['raumtyp'].apply(determine_area_type)
regionclusters['area_type'] = regionclusters['raumtyp'].astype(str).map(category_dict)

# Explode the multipolygons into individual polygons
regionclusters_split = regionclusters.explode(index_parts=False)

# Reset index to clean up the DataFrame
regionclusters_split.reset_index(drop=True, inplace=True)

# Display the resulting DataFrame
regionclusters_split

# Read the CSV file
folder = "input/"
areas = pd.read_csv( folder + "plz_areas.csv")  

# Convert the 'WKT' column to Shapely MultiPolygon geometry
areas['geometry'] = areas['WKT'].apply(lambda wkt_str: wkt.loads(wkt_str))

# Create a GeoDataFrame
gdf_areas = gpd.GeoDataFrame(areas, geometry='geometry')

# Set the coordinate reference system (CRS)
gdf_areas.crs = 'EPSG:25832'  # Set the appropriate CRS if it's different


# Import Network
# MATSim network als Geopandas dataframe inkl. LINESTRINGS einlesen
# network = matsim.read_network('D:\\Hannover Daten\\MatSim\\Network XT\\car_network_encfix.xml.gz').as_geo().set_crs('epsg:25832') # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
# network = matsim.read_network('input/simRes/500-it/basecase_13052025/basecase_13052025.output_network.xml.gz').as_geo().set_crs('epsg:25832')
network = matsim.read_network('input/simRes/BC_500_it_no_reduction/basecase_13052025.output_network.xml.gz').as_geo().set_crs('epsg:25832')
# dict mit Netzwerk Link Längen
# link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
network.head()

# falls vorhanden Ergebnisse der nächsten Zellen laden, sonst überspringen
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)
with open('input/networkplus.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    networkplus = pd.read_pickle(pickle_file)     




C:\Users\bienzeisler\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)
C:\Users\bienzeisler\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


In [3]:
# Mapping: ID -> ausgeschriebener Typ
CATEGORY_NAME_BY_ID = {
    1: "Metropolitan Center",
    2: "High-Density Residential Use",
    3: "Dense Mixed Use",
    4: "Residential Use",
    5: "Industrial Use",
    6: "Urbanized Periphery",
    7: "Rural with Industrial Influence",
    8: "Rural without Industrial Influence",
}

# Reverse: Name -> ID (falls du mal Namen reinbekommst)
CATEGORY_ID_BY_NAME = {v: k for k, v in CATEGORY_NAME_BY_ID.items()}

# Gruppierung (1–3 Urban, 4–6 Suburban, 7–8 Rural)
GROUP_BY_ID = {
    1: "Urban", 2: "Urban", 3: "Urban",
    4: "Suburban", 5: "Suburban", 6: "Suburban",
    7: "Rural", 8: "Rural",
}
GROUP_BY_NAME = {CATEGORY_NAME_BY_ID[i]: g for i, g in GROUP_BY_ID.items()}

def area_type_name(raumtyp):
    """Gibt den ausgeschriebenen 8er-Typ für raumtyp (int/str) zurück."""
    try:
        rid = int(str(raumtyp).strip())
    except Exception:
        return "unknown"
    return CATEGORY_NAME_BY_ID.get(rid, "unknown")

def area_type_group(x):
    """Gibt die Gruppe (Urban/Suburban/Rural) für ID oder Namen zurück."""
    # Versuche ID
    try:
        rid = int(str(x).strip())
        if rid in GROUP_BY_ID:
            return GROUP_BY_ID[rid]
    except Exception:
        pass
    # Sonst als Name interpretieren
    name = str(x).strip()
    return GROUP_BY_NAME.get(name, "unknown")


In [4]:
# Create a color column based on freespeed and lane conditions
def assign_type(row):
    if row['freespeed_kmh'] <= 50:
        return 'urban'
    elif row['freespeed_kmh'] > 50 and row['permlanes'] >= 2:
        return 'highway'
    else:
        return 'rural'  # or any other default color

network['freespeed_kmh'] = network['freespeed'] * 3.6
network['road_type'] = network.apply(assign_type, axis=1)

link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
link_geometry = network[['link_id', 'geometry']].set_index('link_id').to_dict()['geometry']
link_type = network[['link_id', 'road_type']].set_index('link_id').to_dict()['road_type']
# dict mit Netzwerk Link Raumtypen
link_raumtyp = networkplus['raumtyp'].to_dict()

# Create a dictionary linking each area type code to its description
area_type_dict = {
    1: "Metropolitan Center",
    2: "High-Density Residential Use",
    3: "Dense Mixed Use",
    4: "Residential Use",
    5: "Industrial Use",
    6: "Urbanized Periphery",
    7: "Rural with Industrial Influence",
    8: "Rural without Industrial Influence"
}

from matplotlib.colors import to_hex
# Define the desired order of legend labels in English
# Create a dictionary linking each category to a color



category_color_dict = {
    "Metropolitan Center": "#482878",
    "High-Density Residential Use": "#3f4989",
    "Dense Mixed Use": "#31688e",
    "Residential Use": "#26828e",
    "Industrial Use": "#1f9e89",
    "Urbanized Periphery": "#35b779",
    "Rural with Industrial Influence": "#6fce58",
    "Rural without Industrial Influence": "#b5de2b"
}

category_color_dict_agg = {
    'Urban': '#808080',      # Grau
    'Suburban': '#1f78b4',  # Blau
    'Rural': '#33a02c'       # Grün
}



category_color_dict_num = {
    0: "#482878",
    1: "#3f4989",
    2: "#31688e",
    3: "#26828e",
    4: "#1f9e89",
    5: "#35b779",
    6: "#6fce58",
    7: "#b5de2b"
}



# Create a dictionary linking each category to a color
category_color = {category: i / (len(category_color_dict) - 1) for i, category in enumerate(category_color_dict)}

# Create the colormap
cmap = LinearSegmentedColormap.from_list("custom_cmap", sns.color_palette("viridis_r", len(category_color_dict))[::-1])

# Convert the colormap to a dictionary
palette_dict = {cat: to_hex(cmap(category_color[cat])) for cat in category_color_dict}


In [5]:
# def Methoden

def parse_events(event_file):
    """
    This function parses events from a given event file. It filters out events of type 'left link' and 'actstart'.
    It also counts the number of events for different types of vehicles and stores the last link and time for each vehicle.
    """
    
    # Filter events of interest
    events = matsim.event_reader(event_file, types='left link,actstart')

    # Consolidated link counts and vehicle tours
    link_counts = defaultdict(lambda: defaultdict(int))
    vehicle_tour = defaultdict(list)  # This will now store (link, time) tuples
    service_events = defaultdict(list)
    last_link = defaultdict(str)
    
    # Helper function to update link counts
    def update_link_counts(vehicle_type, link, time):
        if link != last_link[vehicle]:
            link_counts[link][vehicle_type] += 1
            vehicle_tour[vehicle].append((link, time))  # Store the link and time as a tuple
            last_link[vehicle] = link
    
    nr_events = 0
    for event in events:        
        if event['time'] > DAYEND: continue
            
        nr_events =  nr_events + 1
        
        if event['type'] == 'left link':
            vehicle = event['vehicle']
            if '_Supply_Vehicle_' in vehicle or '_veh_supply_' in vehicle:
                if "supply_light_van" in vehicle:
                    update_link_counts('supply_van', event['link'], event['time'])
                elif "light" in vehicle:
                    update_link_counts('truck_light_count', event['link'], event['time'])
                else:
                    update_link_counts('truck_count', event['link'], event['time'])

            elif '_CEP_Vehicle_' in vehicle or '_veh_cep_' in vehicle or '_egrocery_van_' in vehicle:
                update_link_counts('van_count', event['link'], event['time'])
                if "size_m" in vehicle:
                    update_link_counts('m_count', event['link'], event['time'])
                elif "size_xl" in vehicle:
                    update_link_counts('xl_count', event['link'], event['time'])
            elif '_cargoBike_' in vehicle or '_cargobike_' in vehicle:
                update_link_counts('bike_count', event['link'], event['time'])
            
        elif event['type'] == 'actstart' and event['actType'] in ['end', 'service']:
            person = event['person']
            if event['actType'] == 'end':
                if any(keyword in person for keyword in ['_Supply_Vehicle_', '_veh_supply_', '_CEP_Vehicle_', '_veh_cep_', '_egrocery_van_', '_cargoBike_', '_cargobike_']):
                    vehicle_tour[person].append((event['link'], event['time']))  # Store the link and time as a tuple
            if event['actType'] == 'service':
                service_events[person].append(event['link'])  # No need to store time for service events as per your original code
                
    
    # Convert link counts to DataFrame and handle missing values
    df = pd.DataFrame(link_counts).transpose()
    df.fillna(0, inplace=True)

    # Ensure columns exist in DataFrame
    for col in ['truck_count', 'truck_light_count', "supply_van_count", 'van_count', 'bike_count']:
        if col not in df.columns:
            df[col] = 0
     
    # Explicitly ensure the 'bike_count' column exists
    if 'bike_count' not in df.columns:
        df['bike_count'] = 0
        
    df['total_count'] = df['truck_count'] + df['truck_light_count'] + df['supply_van_count'] + df['van_count'] + df['bike_count']
        
    df['total_count_supply'] = df['truck_count'] + df['truck_light_count'] + df['supply_van_count'] 
    
    network_volumes = network.merge(df, left_on='link_id', right_index=True, how='left')
    network_volumes.fillna(0, inplace=True)
    
    return vehicle_tour, service_events, network_volumes, nr_events

# vehicle string pattern
# def parse_events(event_file):
#     """
#     This function parses events from a given event file. It filters out events of type 'left link' and 'actstart'.
#     It also counts the number of events for different types of vehicles and stores the last link for each vehicle.
    
#     Args:
#         event_file (str): The path to the event file to be parsed.
        
#     Returns:
#         vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
#         service_events (dict): A dictionary mapping each person to a list of service events.
#         network_volumes (DataFrame): A DataFrame containing the counts of each type of vehicle on each link.
#     """

#     # Helper function to update link counts
#     def update_link_counts(vehicle_type, link, time):
#         if link != last_link[vehicle]:
#             link_counts[link][vehicle_type] += 1
#             vehicle_tour[vehicle].append((link, time))  # Store the link and time as a tuple
#             last_link[vehicle] = link
        
#     # Only returns events of type 'left link' and 'actstart:
#     events = matsim.event_reader(event_file, types='left link,actstart')

#     # defaultdict creates a blank dict entry on first reference; similar to {} but more friendly
#     link_counts_trucks = defaultdict(int)
#     link_counts_vans = defaultdict(int)
#     link_counts_bikes = defaultdict(int)
#     link_counts_cars = defaultdict(int)
#     link_counts = defaultdict(lambda: defaultdict(int))

#     vehicle_tour = defaultdict(list)
#     service_events = defaultdict(list)
#     last_link = defaultdict(str)

#     nr_events = 0
#     for event in events:        
#         if event['time'] > DAYEND: continue
            
#         nr_events =  nr_events + 1
        
#         if event['type'] == 'left link':
#             vehicle = event['vehicle']
#             if '_Supply_Vehicle_' in vehicle or '_veh_supply_' in vehicle:
#                 if "supply_light_van" in vehicle:
#                     update_link_counts('supply_van', event['link'], event['time'])
#                 elif "light" in vehicle:
#                     update_link_counts('truck_light_count', event['link'], event['time'])
#                 else:
#                     update_link_counts('truck_count', event['link'], event['time'])

#             elif '_CEP_Vehicle_' in vehicle or '_veh_cep_' in vehicle or '_egrocery_van_' in vehicle:
#                 update_link_counts('van_count', event['link'], event['time'])
#                 if "size_m" in vehicle:
#                     update_link_counts('m_count', event['link'], event['time'])
#                 elif "size_l" in vehicle:
#                     update_link_counts('xl_count', event['link'], event['time'])
#             elif '_cargoBike_' in vehicle or '_cargobike_' in vehicle:
#                 update_link_counts('bike_count', event['link'], event['time'])
            
#         elif event['type'] == 'actstart' and event['actType'] in ['end', 'service']:
#             person = event['person']
#             if event['actType'] == 'end':
#                 if any(keyword in person for keyword in ['_Supply_Vehicle_', '_veh_supply_', '_CEP_Vehicle_', '_veh_cep_', '_egrocery_van_', '_cargoBike_', '_cargobike_']):
#                     vehicle_tour[person].append((event['link'], event['time']))  # Store the link and time as a tuple
#             if event['actType'] == 'service':
#                 service_events[person].append(event['link'])  # No need to store time for service events as per your original code
    


#     # convert link_counts dict to a pandas dataframe
#     link_counts_trucks = pd.DataFrame.from_dict(link_counts_trucks, orient='index', columns=['truck_count']).rename_axis('link_id')
#     link_counts_vans = pd.DataFrame.from_dict(link_counts_vans, orient='index', columns=['van_count']).rename_axis('link_id')
#     link_counts_bikes = pd.DataFrame.from_dict(link_counts_bikes, orient='index', columns=['bike_count']).rename_axis('link_id')
#     link_counts_cars = pd.DataFrame.from_dict(link_counts_cars, orient='index', columns=['car_count']).rename_axis('link_id')

#     # attach counts to our Geopandas network from above
#     network_volumes = network.merge(link_counts_trucks, on='link_id', how='left').merge(link_counts_vans, on='link_id', how='left').merge(link_counts_bikes, on='link_id', how='left').merge(link_counts_cars, on='link_id', how='left')
#     network_volumes['car_count'] = network_volumes['car_count'].fillna(0)
#     network_volumes['truck_count'] = network_volumes['truck_count'].fillna(0)
#     network_volumes['van_count'] = network_volumes['van_count'].fillna(0)
#     network_volumes['bike_count'] = network_volumes['bike_count'].fillna(0)
#     network_volumes['total_count'] = network_volumes['truck_count'] + network_volumes['van_count'] + network_volumes['bike_count']

#     return vehicle_tour, service_events, network_volumes, nr_events


def vehicle_stats(vehicle_tour, service_events):
    """
    This function calculates statistics for each vehicle, including the number of services, total tour length,
    distance to the first service, and distribution of services over different types of areas.
    
    Args:
        vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
        service_events (dict): A dictionary mapping each person to a list of service events.
        
    Returns:
        veh_df (DataFrame): A DataFrame containing statistics for each vehicle.
    """
        
    # init Ergebnis Objekte
    vehicle_stats = list()

    # Loop über Fahrzeuge und ihre Services
    for veh, ser in service_events.items():
        # Unterscheidung Fahrzeugtypen
        if '_Supply_Vehicle_' in veh or '_veh_supply_' in veh:
            c = 'truck'            
        if '_CEP_Vehicle_' in veh or '_veh_cep_' in veh or '_egrocery_van_' in veh:
            c = 'van'
        if '_cargoBike_' in veh or '_cargobike_' in veh:
            c = 'bike'
        
        # tourlen = 0
        # firstservicedist = None
        # # Loop über alle Tourlinks des Fahrzeuges
        # for link in vehicle_tour[veh]:
        #     # falls erster Service erreicht, Strecke bis dahin speichern
        #     if link == ser[0] and firstservicedist is None:
        #         firstservicedist = tourlen
        #     # Streckenlänge aufsummieren
        #     tourlen += link_length[link]
        # # Sonderfälle abfangen, wo ein Service kurz vor Tagesende beginnt und am nächsten Tag weitergefahren wird
        # if firstservicedist is None: firstservicedist = tourlen

        tourlen = 0
        firstservicedist = None
        service_dists = dict()

        if not ser:
            continue

        for link, time in vehicle_tour[veh]: 
            # Strecke aufsummieren
            link_km = link_length[link] / 1000  # Direkt km
            tourlen += link_km
            for s in ser:
                # Sobald Servicepunkt erreicht (einmalig erfassen)
                if link == s and s not in service_dists:
                    service_dists[s] = tourlen
                    if firstservicedist is None:
                        firstservicedist = tourlen  # Speichere ersten Treffer

        # Fallback: falls ein Service nicht in der Tour auftaucht
        for s in ser:
            if s not in service_dists:
                service_dists[s] = tourlen
        if firstservicedist is None:
            firstservicedist = tourlen
           
        indexService = 0
        start_service_distance_count = False
        currentServiceDist = 0
        serviceDistList= []    

        # Service Verteilung über Raumtypen ermitteln
        raumtypen_services = defaultdict(int)
        for s in ser:
            rt = link_raumtyp.get(s, 0) # keinem Raumtyp zugeordnet -> 0
            raumtypen_services[rt] += 1        
    

        summ_service_distances = 0
        previous_time = 0     


        vehicle_stats.append([
            veh, c, len(ser),
            tourlen ,
            firstservicedist ,
            raumtypen_services,
            service_dists,
            firstservicedist   # explizite Spalte für initial_delivery_distance
        ])


    # Dataframe aus Liste
    veh_df = pd.DataFrame(vehicle_stats, columns=[
        'vehicle_id', 'veh_class', 'service_num',
        'tour_km', 'first_service_dist',
        'raumsplit', 'service_dists',
        'initial_delivery_distance'
    ])

    return veh_df

# Definition of help-methods needed for the convertion process

# Method returning the index of an element of a dictionary
def get_nth_key(dictionary, n=0):
    """
    This function returns the nth key of a dictionary.
    
    Args:
        dictionary (dict): The dictionary to get the key from.
        n (int): The index of the key to get. Default is 0.
        
    Returns:
        key: The nth key of the dictionary.
    """
        
    if n < 0:
        n += len(dictionary)
    for i, key in enumerate(dictionary.keys()):
        if i == n:
            return key
    raise IndexError("dictionary index out of range") 

    
# Get Logistic Provider from XML ID
def getProviderFromID(carrierID):
    """
    This function returns the provider name based on the carrier ID.
    
    Args:
        carrierID (str): The ID of the carrier.
        
    Returns:
        str: The name of the provider.
    """
        
    if("dhl" in str(carrierID)):
        return "dhl"  
    elif("amazon" in str(carrierID)):
        return "amazon"  
    elif("ups" in str(carrierID)):
        return "ups"  
    elif("gls" in str(carrierID)):
        return "gls"  
    elif("dpd" in str(carrierID)):
        return "dpd"  
    elif("fedex" in str(carrierID)):
        return "fedex"  
    elif("hermes" in str(carrierID)):
        return "hermes"  
    elif("wl" in str(carrierID)):
        return "White-Label"  
    else:
        raise ValueError('Carrier Provider unknown: ' + str(carrierID))
        
# Definition of needen Classes: Carrier, Vehicle, Plan & Service with Variables

# A carrier object has an ID and dictionaries with all Vehicles and all Services
class Carrier:  
    """
    This class represents a Carrier with an ID and dictionaries with all Vehicles and all Services.
    """
        
    def __init__(self, carrierID): 
        self.carrierID = carrierID 
        self.vehicles = {} 
        self.services = {}   
        self.missedDeliveries = []
        self.logisticProvider = getProviderFromID(self.carrierID)
        
    def __str__(self):
        return "[Carrier ID: " + str(self.carrierID) + " with " + str(len(self.vehicles)) + " Vehicles and " + str(len(self.services)) + " Services]"
    
    def __repr__(self):
        return self.__str__()
    
    def getNumberOfVehicles(self):
        return (len(self.vehicles))
    
    def getNumberOfServices(self):
        return (len(self.services))
        
    def addVehicle(self, vehicle): 
        self.vehicles[vehicle.getVehicleId] = vehicle
        
    def addService(self, service): 
        self.services[service.getServiceID] = service

    def getTotalNumberofServices(self):  
        return sum((s.getDemand() for s in self.services.values()))
    
    def getProvider(self):
        return self.logisticProvider
    
    def getServices(self): 
        return [s for s in self.services.values()]   
    
    def getCarrierId(self): 
        return self.carrierID    
    
    def getVehicles(self): 
        return self.vehicles
    
    def getMissedDeliveries(self): 
        return self.missedDeliveries  
    
    def setMissedDeliveries(self, missDeliveries): 
        self.missedDeliveries = missDeliveries


# A Vehicle has an ID, a Typ and you can add Plans to this vehicles with services and Routes. 
# These Vehicles / Plans have to be converted to Sumo! 
class Vehicle: 
    """
    This class represents a Vehicle with an ID, a Type and you can add Plans to this vehicles with services and Routes.
    """
        
    def __init__(self, vehicleID, vehicleType): 
        self.vehicleID = vehicleID 
        self.vehicleType = vehicleType 
        self.plans = []
        
    def __str__(self):
        return "[Vehicle ID: " + str(self.vehicleID) + " with Type: " + self.vehicleType +"]"
    
    def __repr__(self):
        return self.__str__()
    
    def addVehiclePlan(self, plan): 
        self.plans.append(plan)
        
    def changeVehicleType(self, vehicleType): 
        self.vehicleType == vehicleType
        
    def getVehicleId(self): 
        return self.vehicleID
    
    def getPlans(self):
        return self.plans


# A Plan is a sequence of Activies (start, service, service, ... , end)
# all important Information are stored in the activities / legs dictionary
# The Order of the sequence is stored as a list called "planSequence"
class Plan:  
    """
    This class represents a Plan which is a sequence of Activities (start, service, service, ... , end).
    """
        
    def __init__(self, planId, vehicle, activities, legs): 
        self.planId = planId
        self.vehicle = vehicle
        self.activities = activities 
        self.legs = legs 
        self.planSequence = [] 
        
    def __str__(self):
        return "[Plan ID: " + str(self.planId) + " with " + str(len(self.activities)) +" Activities and " + str(len(self.legs)) + " Legs]"
    
    def __repr__(self):
        return self.__str__()
    
    def getPlanSequence(self):
        return self.planSequence
    
    def createInternalPlanSequence(self):
        for i in range (len(self.legs)):            
            self.planSequence.append(get_nth_key(self.activities, i))
            self.planSequence.append(get_nth_key(self.legs, i))
        self.planSequence.append(get_nth_key(self.activities, len(self.activities)-1))    

# A Service is a data container storing all available information from the CSV/XML
class Service:
    """
    Data container for a delivery/pickup service.
    All durations are stored in seconds (int).
    Extra attributes are kept in a dict.
    """

    def __init__(self, serviceType, serviceID, capacityDemand, duration, link, extra_attributes=None):
        self.serviceType = serviceType
        self.serviceID = serviceID
        self.capacityDemand = capacityDemand
        # ensure duration is numeric seconds
        try:
            self.duration = int(duration)
        except Exception:
            self.duration = 0
        self.link = link
        self.extra_attributes = extra_attributes or {}

        # optional: vehicleType placeholder if you need it later
        self.vehicleType = self.extra_attributes.get("vehicleType", None)

    def __str__(self):
        return f"[Service ID: {self.serviceID} (Type: {self.serviceType}) with a CapacityDemand of {self.capacityDemand} to Link: {self.link}]"

    __repr__ = __str__

    # --- attributes API ---
    def getAttribute(self, key, default=None):
        return self.extra_attributes.get(key, default)

    def setAttribute(self, key, value):
        self.extra_attributes[key] = value

    # --- typed getters/setters ---
    def getServiceType(self):
        return self.serviceType

    def changeVehicleType(self, vehicleType):
        # was '==' before (no effect). Should assign.
        self.vehicleType = vehicleType
        self.extra_attributes["vehicleType"] = vehicleType

    def getDemand(self):
        return self.capacityDemand

    def getServiceID(self):
        return self.serviceID

    def getServiceLink(self):
        return self.link

    def setServiceLink(self, link):
        self.link = link

    def getDuration(self):
        """Return dwell time in seconds."""
        return int(self.duration)

    def setDuration(self, duration_s):
        """Set dwell time in seconds."""
        self.duration = int(duration_s)
    
def extract_provider(vehicle_id):
    """
    This function extracts the provider from the vehicle ID.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The name of the provider.
    """
        
    return vehicle_id.split("_")[1]

def extract_veh_size(vehicle_id):
    """
    Extracts the vehicle size from the vehicle ID by looking for 'size_' and returning the next token.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The vehicle size (e.g., 'l', 'm', 'l'), or 'unknown' if not found.
    """
    try:
        parts = vehicle_id.split("size_")
        if len(parts) > 1:
            return parts[1].split("_")[0]
        else:
            return "unknown"
    except Exception as e:
        print(f"Error extracting vehicle size from '{vehicle_id}': {e}")
        return "unknown"

def extract_main_area_type(raumsplit):
    """
    This function extracts the main area type from the raumsplit.
    
    Args:
        raumsplit (dict): The dictionary containing the raumsplit.
        
    Returns:
        str: The main area type.
    """
        
    return max(raumsplit.items(), key=lambda x: x[1])[0]

def process_vehicle_data(veh_df_van):
    """
    This function processes the vehicle data and adds provider, vehicle size, and main area type to the DataFrame.
    
    Args:
        veh_df_van (DataFrame): The DataFrame containing the vehicle data.
        
    Returns:
        DataFrame: The processed DataFrame.
    """
        
    veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
    veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
    veh_df_van['main_area_type'] = veh_df_van['raumsplit'].apply(extract_main_area_type)

    return veh_df_van

def get_vehicles(vehicle_tour):
    """
    This function gets the vehicles from the vehicle tour.
    
    Args:
        vehicle_tour (dict): The dictionary containing the vehicle tour.
        
    Returns:
        list: The list of vehicles.
    """
        
    return [vehicle for vehicle in vehicle_tour if "_supply_" not in vehicle]

def create_plot_data(vehicles, event_file):
    """
    This function creates the plot data.
    
    Args:
        vehicles (list): The list of vehicles.
        event_file (str): The path to the event file.
        
    Returns:
        DataFrame: The DataFrame containing the plot data.
    """
        
    plot_data = pd.DataFrame({
    "Start time" : [0],
    "End time" : [0],
    "Tour Duration" : [0],
    "Service Duration" : [0],
    "Travel Duration" : [0],
    }, index=[vehicles,])

    #calculate start and end time
    events = matsim.event_reader(event_file, types='actstart,actend')
    event_lists = { 'start': [], 'end': [] }

    for event in events:
        if "_supply_" in event["person"]:
            continue
        if event['actType'] == "start" and event['type'] == "actend":
            event_lists['start'].append(event)
        if event['actType'] == "end" and event['type'] == "actstart":
            event_lists['end'].append(event)
    df_start = pd.DataFrame(event_lists['start'])
    df_end = pd.DataFrame(event_lists['end'])

    for event in range(len(df_start)):
            plot_data["Start time"][df_start.iloc[event]["person"]] = df_start.iloc[event]['time']
    for event in range(len(df_end)):
            plot_data["End time"][df_end.iloc[event]["person"]] = df_end.iloc[event]['time']


    #calculate travel time - need to remove the service time
    events = matsim.event_reader(event_file, types='actstart,actend')
    event_lists = { 'start': [], 'end': [] }

    for event in events:
        if "_supply_" in event["person"]:
            continue
        if event['actType'] == "service" and event['type'] == "actstart":
            event_lists['start'].append(event)   # Beginn Service
        if event['actType'] == "service" and event['type'] == "actend":
            event_lists['end'].append(event)     # Ende Service
    df_start = pd.DataFrame(event_lists['start'])
    df_end = pd.DataFrame(event_lists['end'])

    for row in range(len(plot_data)):
        plot_data.iloc[row]["Tour Duration"] = plot_data.iloc[row]["End time"] - (plot_data.iloc[row]["Start time"])

    startG = df_start.groupby("person")
    endG = df_end.groupby("person")

    for n,g in startG:
        time = 0

        g = g.reset_index(drop=True)   

        g2 = endG.get_group(n)
        g2 = g2.reset_index(drop=True)

        result = pd.merge(g, g2, left_index=True, right_index=True)

        for i in range(len(result)):
            time = time + (result.loc[i]["time_y"] - result.loc[i]["time_x"])

            plot_data.loc[n, "Service Duration"] = abs(time)

    for row in range(len(plot_data)):
        plot_data.iloc[row]["Travel Duration"] = plot_data.iloc[row]["Tour Duration"] - (plot_data.iloc[row]["Service Duration"])

    plot_data = plot_data.astype(str)
    # for row in range(len(plot_data)):
    #     plot_data.iloc[row]["Start time"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Start time"])))
    #     plot_data.iloc[row]["End time"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["End time"])))
    #     plot_data.iloc[row]["Tour Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Tour Duration"])))
    #     plot_data.iloc[row]["Service Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Service Duration"])))
    #     plot_data.iloc[row]["Travel Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Travel Duration"])))



    plot_data.reset_index(inplace=True)
    plot_data = plot_data.rename(columns={'level_0': 'vehicle_id'})

    plot_data["Start time formatted"] = pd.to_datetime(plot_data['Start time'], unit='s')
    plot_data["End time formatted"] = pd.to_datetime(plot_data['End time'], unit='s')
    plot_data["Tour Duration formatted"] = pd.to_datetime(plot_data['Tour Duration'], unit='s')
    plot_data["Service Duration formatted"] = pd.to_datetime(plot_data['Service Duration'], unit='s')
    plot_data["Travel Duration formatted"] = pd.to_datetime(plot_data['Travel Duration'], unit='s')

    plot_data["Start time"] = pd.to_numeric(plot_data["Start time"])
    plot_data["End time"] = pd.to_numeric(plot_data["End time"])
    plot_data["Tour Duration"] = pd.to_numeric(plot_data["Tour Duration"])
    plot_data["Service Duration"] = pd.to_numeric(plot_data["Service Duration"])
    plot_data["Travel Duration"] = pd.to_numeric(plot_data["Travel Duration"])
    
    return plot_data , startG , endG

def parse_carriers_from_xml(root):
    """
    This function parses carriers from an XML root.
    
    Args:
        root (ElementTree): The XML root to parse carriers from.
        
    Returns:
        list: The list of carriers.
    """

    carriers = []
    totalVeh = 0
    ns = {"m": "http://www.matsim.org/files/dtd"}

    for carrierXML in root.findall("m:carrier", ns):
        carrierID = carrierXML.attrib.get("id")
        if "supply" in carrierID:
            continue

        newCarrier = Carrier(carrierID)

        for attributeXML in carrierXML.findall("m:attributes", ns):
            for attribute in attributeXML.findall("m:attribute", ns):
                if attribute.attrib.get("name") == "missedParcelDeliveriesAsString":
                    missedParcels = attribute.text
                    missedParcels = missedParcels.replace("[", "").replace("]", "").replace(" ", "")
                    missedServicePerCarrier = missedParcels.split(",")
                    newCarrier.setMissedDeliveries(missedServicePerCarrier)

        for serviceXML in carrierXML.findall(".//m:service", ns):
            serviceID = serviceXML.attrib.get("id")

            serviceCapacityDemand = int(serviceXML.attrib.get("capacityDemand"))
            serviceDuration_str = serviceXML.attrib.get("serviceDuration")
            serviceDuration = None
            if serviceDuration_str is not None:
                try:
                    # Normal services: format "hh:mm:ss" -> convert to seconds
                    if ":" in serviceDuration_str:
                        h, m, s = map(int, serviceDuration_str.split(":"))
                        serviceDuration = h*3600 + m*60 + s
                    else:
                        # Already numeric, e.g. "360"
                        serviceDuration = int(serviceDuration_str)
                except Exception as e:
                    print(f"Error parsing serviceDuration={serviceDuration_str}: {e}")
                    serviceDuration = 0
            serviceLink = serviceXML.attrib.get("to")

            known_keys = {"id", "capacityDemand", "serviceDuration", "to"}
            extra_attrs = {k: v for k, v in serviceXML.attrib.items() if k not in known_keys}

            attributes_element = serviceXML.find("m:attributes", ns)
            if attributes_element is not None:
                for attr in attributes_element.findall("m:attribute", ns):
                    key = attr.attrib.get("name")
                    value = attr.text
                    if key in ["b2b", "b2c"]:
                        try:
                            value = int(value)
                        except ValueError:
                            value = 0
                    extra_attrs[key] = value

            newService = Service(
                "service",
                serviceID,
                serviceCapacityDemand,
                serviceDuration,
                serviceLink,
                extra_attributes=extra_attrs
            )
            newCarrier.addService(newService)

        for plan in carrierXML.findall(".//m:plan", ns):
            if plan.attrib.get("selected") == "true":
                i = 0
                for tour in plan.findall(".//m:tour", ns):
                    i += 1
                    vehID = tour.attrib.get("vehicleId") + "_" + str(i)
                    vehicle = Vehicle(vehID, "cep")                    

                    b = 0
                    c = 0

                    activities = {}
                    legs = {}
                    routes = {}

                    for act in tour.findall(".//m:act", ns):
                        actType = act.attrib.get("type")
                        if actType == "start":
                            activities[actType] = act
                        elif actType == "end":
                            activities[actType] = act
                        else:
                            actID = act.attrib.get("serviceId")
                            activities[actID] = act

                    for leg in tour.findall(".//m:leg", ns):
                        legs["leg_" + str(b)] = leg
                        b += 1

                    for route in tour.findall(".//m:route", ns):
                        if route.text is None:
                            routes["route_" + str(c)] = None
                        else:
                            routes["route_" + str(c)] = route.text.split(" ")
                        c += 1

                    for key, value in legs.items():
                        routeKey = "route_" + str(key.split("_")[1])
                        route = routes.get(routeKey)
                        value.attrib["route"] = route

                    vehiclePlan = Plan("plan_" + vehID, vehicle, activities, legs)
                    vehiclePlan.createInternalPlanSequence()
                    vehicle.addVehiclePlan(vehiclePlan)
                    newCarrier.addVehicle(vehicle)

                totalVeh += newCarrier.getNumberOfVehicles()

        carriers.append(newCarrier)

    return carriers



def calculate_costs(row):
    veh_time_cost = 22.87 / 3600  # Kosten pro Sekunde

    if "size_l" in row['vehicle_id']:
        veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
    elif "size_m" in row['vehicle_id']: 
        veh_cap, veh_fix, veh_km_cost = 165, 171.78, 0.372
    else:
        if "supply_light_van" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
        elif "light" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 1000, 550.63, 0.48643
        else:
            veh_cap, veh_fix, veh_km_cost = 2000, 618.55, 0.555126     
      
     
    vehicle_fix_cost = veh_fix
    vehicle_km_cost = row["tour_km"] * veh_km_cost
    vehicle_time_cost = row["Tour Duration"] * veh_time_cost

    overtime_seconds = max(0, row["Tour Duration"] - (7.5 * 3600))
    overtime_cost = overtime_seconds * veh_time_cost

    vehicle_cost = vehicle_fix_cost + vehicle_km_cost + overtime_cost

    return pd.Series([vehicle_fix_cost, vehicle_km_cost, vehicle_time_cost, overtime_cost, vehicle_cost])


def add_vehicle_demand_to_result(carriers, result):
    
    """
    This function adds vehicle demand to the result DataFrame.
    
    Args:
        carriers (list):The list of carriers.
        result (DataFrame): The DataFrame to add vehicle demand to.
        
    Returns:
        DataFrame: The DataFrame with added vehicle demand.
    """

    required_cols = [
        'deliveries', 'missed deliveries', 'b2b_ration', 'b2c_ration',
        'ration_check', 'vehicle_load_factor', 'vehicle_deliver_factor',
        'vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost',
        'overtime_cost', 'vehicle_cost', 'service_num'
    ]
    for col in required_cols:
        if col not in result.columns:
            result[col] = np.nan
    expected_ids = []
    
        
    veh = 0
    fail = 0

    # 1. Create a new dictionary to hold vehicle-service mappings
    vehicle_services_dict = {}

    for c in carriers:
        missedDeliveries = c.getMissedDeliveries()
        missed_deliveries_set = set(missedDeliveries) 

        services = c.getServices()

        # print(f"🔍 Carrier {c.getCarrierId()} has {len(c.getVehicles())} vehicles")
        
        
        for k, v in c.getVehicles().items():
            veh = veh + 1 
            result_df_id = "freight_" + c.getCarrierId()+"_veh_"+v.getVehicleId()     
            expected_ids.append(result_df_id)   
         
            veh_df_res = result[result.vehicle_id == result_df_id]    
            if (len(veh_df_res) != 1):
                print(result_df_id)
                fail = fail + 1    
                
            mask = result.vehicle_id == result_df_id
            if mask.sum() == 0:
                print(f"⚠️ ID not found in result: {result_df_id}")   
            
            totalVehDemand = 0            
            missedParcels = 0

            b2b = 0
            b2c = 0

            # 2. Extract services for this vehicle
            veh_services = []
            services_dict = {s.getServiceID(): s for s in services}

            for a in v.getPlans()[0].activities:
                if "service" in a:
                    for s in services:
                        if a == s.getServiceID():
                            service = services_dict.get(a)
                            veh_services.append(service)

                            service_id = s.getServiceID()
                            service_b2b = int(s.getAttribute("b2b", 0))
                            service_b2c = int(s.getAttribute("b2c", 0))

                            # Nachfrage aufsummieren
                            b2b += service_b2b
                            b2c += service_b2c
                            serviceDemand = service_b2b + service_b2c
                            totalVehDemand += serviceDemand


                            merged = s.getAttribute("mergedMetadata", None)
                            
                            if service_id in missed_deliveries_set:
                                missedParcels += serviceDemand  
                            elif merged:
                                try:
                                    merged_dict = json.loads(merged)

                                    # Validierung: mixed muss zu merged passen
                                    if "MIXED" not in service_id.upper():
                                        print(f"⚠️ Warning: mergedMetadata found, but service ID does not indicate MIXED: {service_id}")

                                    for sid, md in merged_dict.items():
                                        if sid in missed_deliveries_set:
                                            # Sub-service demand: prefer capacity, else len(weights), else 1
                                            sub_cap_raw = md.get("capacity", 0)
                                            try:
                                                sub_cap = int(sub_cap_raw)
                                            except Exception:
                                                sub_cap = 0

                                            sub_weights_raw = md.get("weights", [])
                                            # Normalize weights to a list
                                            if isinstance(sub_weights_raw, str):
                                                try:
                                                    parsed = json.loads(sub_weights_raw)
                                                    sub_weights = parsed if isinstance(parsed, list) else [parsed]
                                                except Exception:
                                                    sub_weights = [sub_weights_raw]
                                            elif isinstance(sub_weights_raw, (list, tuple)):
                                                sub_weights = list(sub_weights_raw)
                                            else:
                                                # float/None/other
                                                sub_weights = []

                                            # Assert consistency if both present
                                            if sub_cap > 0 and len(sub_weights) > 0:
                                                assert sub_cap == len(sub_weights), (
                                                    f"Mismatch in merged service {sid}: "
                                                    f"capacity={sub_cap}, len(weights)={len(sub_weights)}"
                                                )

                                            # Final sub-demand
                                            sub_demand = sub_cap if sub_cap > 0 else (len(sub_weights) if len(sub_weights) > 0 else 1)

                                            # Count only missed sub-services
                                            if sid in missed_deliveries_set:
                                                missedParcels += sub_demand
                                except Exception as e:
                                    print(f"⚠️ Error while parsing mergedMetadata for {service_id}: {e}")


                                
                                    
            vehicle_services_dict[result_df_id] = veh_services                         
            veh_cap = 230 if "size_l" in result_df_id else 165

            b2b_ration = b2b / totalVehDemand
            b2c_ration = b2c / totalVehDemand
            
            result.loc[result.vehicle_id == result_df_id, 'b2b_ration'] = b2b_ration
            result.loc[result.vehicle_id == result_df_id, 'b2c_ration'] = b2c_ration
            result.loc[result.vehicle_id == result_df_id, 'ration_check'] = b2b_ration + b2c_ration

            result.loc[result.vehicle_id == result_df_id, 'deliveries'] = totalVehDemand
            if result[result.vehicle_id == result_df_id].empty:
                print(f"Assignment failed for: {result_df_id}")
            
            result.loc[result.vehicle_id == result_df_id, 'missed deliveries'] = round(missedParcels,0)
            
             # Anwenden der Funktion auf den DataFrame
            cols = ['vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost', 'overtime_cost', 'vehicle_cost']
            result[cols] = result.apply(calculate_costs, axis=1)
             

            result.loc[result.vehicle_id == result_df_id, 'vehicle_load_factor'] = totalVehDemand / veh_cap
            result.loc[result.vehicle_id == result_df_id, 'vehicle_deliver_factor'] = (totalVehDemand - missedParcels) / totalVehDemand
    
    validate_vehicle_id_assignment(result, expected_ids)
    
    result['deliveries_per_stop'] = result['deliveries'] / result['service_num']
    result['Hannover'] = result['vehicle_id'].apply(lambda x: any(plz in x for plz in plzList))

    result['services'] = result['vehicle_id'].map(vehicle_services_dict)
    
    return result

def validate_vehicle_id_assignment(result, expected_ids):
    """
    Validates whether all expected vehicle IDs are present in the result DataFrame.
    Prints helpful diagnostics for debugging.
    
    Args:
        result (DataFrame): The result DataFrame after add_vehicle_demand_to_result.
        expected_ids (list): List of vehicle_id strings that should exist in result.
    """
    actual_ids = result['vehicle_id'].astype(str).tolist()

    missing_ids = [vid for vid in expected_ids if vid not in actual_ids]
    duplicate_ids = result['vehicle_id'][result['vehicle_id'].duplicated()].unique().tolist()
    empty_ids = result['vehicle_id'].isna().sum()

    print("\n📋 Vehicle ID Validation Report")
    print("──────────────────────────────")
    print(f"🔢 Expected IDs total: {len(expected_ids)}")
    print(f"✅ Found: {len(expected_ids) - len(missing_ids)}")
    print(f"❌ Missing: {len(missing_ids)}")
    print(f"🔁 Duplicates: {len(duplicate_ids)}")
    print(f"⚠️ Empty vehicle_id entries: {empty_ids}")
    
    if missing_ids:
        print("\n❌ Missing IDs (first 10 shown):")
        for mid in missing_ids[:10]:
            print(f"  - {mid}")
    
    if duplicate_ids:
        print("\n🔁 Duplicate vehicle_ids:")
        for did in duplicate_ids:
            print(f"  - {did}")


def attach_service_times_from_events(result_df, startG, endG, snap_1s=True):
    """
    Attach service start/end timestamps to each Service object and optionally
    snap end time to declared duration when the difference is exactly +/-1 s.

    Writes into Service.extra_attributes or via setAttribute:
      - start_ts: ISO 8601 timestamp
      - end_ts: ISO 8601 timestamp
      - post_start_ts: end_ts + 1 second
      - observed_dur_s: float observed duration in seconds
      - snapped: bool whether end_ts was adjusted by snap_1s

    Args:
        result_df: DataFrame with columns ['vehicle_id','services']
        startG, endG: groupby('person') on service actstart/actend events
        snap_1s: if True, when |observed - declared| == 1, set end = start + declared
    """
    if "services" not in result_df.columns:
        raise ValueError("result_df must contain a 'services' column with lists of Service objects.")

    for _, row in result_df.iterrows():
        vid = row["vehicle_id"]
        services = row["services"]
        if not services:
            continue

        try:
            g_start = startG.get_group(vid).sort_values(by="time").reset_index(drop=True)
        except Exception:
            print(f"[attach_service_times] missing start events for {vid}")
            continue
        try:
            g_end = endG.get_group(vid).sort_values(by="time").reset_index(drop=True)
        except Exception:
            print(f"[attach_service_times] missing end events for {vid}")
            continue

        n_pairs = min(len(g_start), len(g_end), len(services))
        if not (len(g_start) == len(g_end) == len(services)):
            print(f"[attach_service_times] count mismatch {vid}: "
                  f"starts={len(g_start)} ends={len(g_end)} services={len(services)} -> using {n_pairs}")

        for i in range(n_pairs):
            svc = services[i]

            start_sec = float(g_start.loc[i, "time"])
            end_sec_raw = float(g_end.loc[i, "time"])

            # declared duration from Service in seconds
            try:
                declared = float(svc.getDuration())
            except Exception:
                declared = None

            # observed duration from events
            observed = end_sec_raw - start_sec

            snapped = False
            end_sec = end_sec_raw

            # Optional snap when exactly off by 1 second
            if snap_1s and declared is not None and abs(observed - declared) == 1.0:
                end_sec = start_sec + declared
                observed = declared
                snapped = True

            # Timestamps with exact second resolution, no rounding
            start_ts = pd.to_datetime(start_sec, unit="s")
            end_ts   = pd.to_datetime(end_sec,   unit="s")
            post_ts  = end_ts + pd.to_timedelta(1, unit="s")

            # Store on service
            try:
                svc.setAttribute("start_ts", start_ts.isoformat())
                svc.setAttribute("end_ts", end_ts.isoformat())
                svc.setAttribute("post_start_ts", post_ts.isoformat())
                svc.setAttribute("observed_dur_s", observed)
                svc.setAttribute("snapped", snapped)
            except AttributeError:
                if hasattr(svc, "extra_attributes") and isinstance(svc.extra_attributes, dict):
                    svc.extra_attributes["start_ts"] = start_ts.isoformat()
                    svc.extra_attributes["end_ts"] = end_ts.isoformat()
                    svc.extra_attributes["post_start_ts"] = post_ts.isoformat()
                    svc.extra_attributes["observed_dur_s"] = observed
                    svc.extra_attributes["snapped"] = snapped
                else:
                    print(f"[attach_service_times] cannot store attributes for {vid}")

    return result_df




In [6]:
def format_runtime(duration):
    """Format the runtime duration in either seconds, milliseconds, or minutes."""
    if duration < 1:  # If less than 1 second
        return f"{duration * 1000:.2f} ms"
    elif duration < 60:  # If less than 1 minute
        return f"{duration:.2f} s"
    else:  # If 1 minute or more
        minutes = duration // 60
        seconds = duration % 60
        return f"{int(minutes)} min {seconds:.2f} s"

## Batch runner and export helpers
Die folgenden Zellen fügen Export-Wrapper und einen Batch-Runner hinzu, um alle Runs unter `input/simRes/batch` in einem Rutsch zu verarbeiten und Artefakte pro Run in `processed/<scenario>/<run_name>` zu speichern.

In [ ]:
import os
import pickle
import traceback
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter


BATCH_DIR = r"C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch"
OUTPUT_BASE = os.path.join(BATCH_DIR, "processed\c")  # write alongside matsim processed

DAYEND = 24*3600

# This is a list of postal codes (it seems) that the script will be working with.
plzList = ["30159", "30161", "30163", "30165", "30167", "30169", "30171", "30173", "30175", "30177", "30179", "30419", "30449"
        ,"30451" ,"30453" ,"30455" ,"30457" ,"30459" ,"30519" ,"30521" ,"30539" ,"30559" ,"30625" ,"30627" ,"30629",
        "30625" , "30627", "30629", "30655", "30657", "30659", "30669", "31303", "31303"]

os.makedirs(OUTPUT_BASE, exist_ok=True)

def _safe_save_obj(obj, path):
    base, ext = os.path.splitext(path)
    pkl_path = base + '.pkl'
    print("Writing Results to: ", pkl_path)
    with open(pkl_path, 'wb') as f:
        pickle.dump(obj, f)
    return pkl_path

class SimpleProgress:
    def __init__(self, total: int):
        import time
        self.total = total
        self.start = time.time()
        self.last_len = 0
    def update(self, i: int, prefix: str = ""):
        import time
        now = time.time()
        elapsed = now - self.start
        done = i
        remaining = max(self.total - done, 0)
        eta = (elapsed / done * remaining) if done else 0
        bar_len = 30
        filled = int(bar_len * done / self.total) if self.total else 0
        bar = "#" * filled + "-" * (bar_len - filled)
        msg = f"{prefix} [{bar}] {done}/{self.total} | elapsed {elapsed:6.1f}s | ETA {eta:6.1f}s"
        print("\r" + msg + " " * max(self.last_len - len(msg), 0), end="")
        self.last_len = len(msg)
        if done == self.total:
            print()

def _discover_runs(batch_dir: str):
    runs = []
    if not os.path.isdir(batch_dir):
        return runs
    for scen_entry in sorted(os.listdir(batch_dir)):
        scen_path = os.path.join(batch_dir, scen_entry)
        if not os.path.isdir(scen_path):
            continue
        scenario = scen_entry.split(" ")[0]
        subs = [os.path.join(scen_path, d) for d in os.listdir(scen_path) if os.path.isdir(os.path.join(scen_path, d))]
        search_roots = subs if subs else [scen_path]
        for root in search_roots:
            try:
                files = os.listdir(root)
            except PermissionError:
                continue
            ev = [f for f in files if f.endswith('output_events.xml') or f.endswith('output_events.xml.gz')]
            ca = [f for f in files if f.endswith('output_carriers.xml') or f.endswith('output_carriers.xml.gz')]
            if not ev or not ca:
                continue
            def prefix_of(fn: str) -> str:
                return fn[:-3] if fn.endswith('.gz') else fn
            ev_pref = {prefix_of(f).replace('.output_events.xml',''): f for f in ev}
            ca_pref = {prefix_of(f).replace('.output_carriers.xml',''): f for f in ca}
            for pref in sorted(set(ev_pref) & set(ca_pref)):
                event_file = os.path.join(root, ev_pref[pref])
                carrier_file = os.path.join(root, ca_pref[pref])
                run_name = os.path.basename(root) if root != scen_path else pref
                out_dir = os.path.join(OUTPUT_BASE, scenario, run_name)
                runs.append({
                    'scenario': scenario,
                    'run_name': run_name,
                    'event_file': event_file,
                    'carrier_file': carrier_file,
                    'out_dir': out_dir,
                })
    return runs

def run_batch():
    
    runs = _discover_runs(BATCH_DIR)
    if not runs:
        print(f"No runs found in {BATCH_DIR}")
        return
    print(f"Discovered {len(runs)} runs across scenarios: {sorted(set([r['scenario'] for r in runs]))}")

    progress = SimpleProgress(total=len(runs))
    rows = []
    for i, r in enumerate(runs, 1):
        # if i > 1:
        #     continue
        
        progress.update(i - 1, prefix=f"Processing {r['scenario']}/{r['run_name']}")
        # os.makedirs(r['out_dir'], exist_ok=True)
        try:
            globals()['OUT_DIR'] = r['out_dir']
            # Persist exports regardless (if computed globals are available)
            try:

                city = regionclusters[regionclusters.raumtyp < 7]

                result_dataframes = {}
                result_networks= {}

                event_file = r['event_file']
                carrier_file = r['carrier_file']

                # Begin script execution
                print(f"[{datetime.datetime.now()}] [1/10] Initiating script execution...")
                overall_start_time = time.time()

                print(f"[{datetime.datetime.now()}] [2/10] Loading events from {event_file}...")
                start_time = time.time()
                vehicle_tour, service_events, network_volumes, nr_events = parse_events(event_file)
                print(f"[{datetime.datetime.now()}] [INFO] Loaded events for {len(vehicle_tour)} vehicles. (Runtime: {format_runtime(time.time() - start_time)})")
                print(f"[{datetime.datetime.now()}] [INFO] Number of parsed events: {nr_events}")
                print(f"[{datetime.datetime.now()}] [INFO] Network size: {len(network_volumes)}")

                print(f"[{datetime.datetime.now()}] [3/10] Processing network volumes...")
                start_time = time.time()
                clipped_network_volumes = gpd.clip(network_volumes, gdf_areas)
                end_time = time.time()
                reduced_size = len(clipped_network_volumes)
                reduction_percentage = (1 - (reduced_size / len(network_volumes))) * 100

                print(f"[{datetime.datetime.now()}] [INFO] Network volumes processed. (Runtime: {format_runtime(end_time - start_time)})")
                print(f"[{datetime.datetime.now()}] [INFO] Reduced Network size to: {reduced_size} ({reduction_percentage:.2f}% reduction)")
                    
                print(f"[{datetime.datetime.now()}] [4/10] Calculating vehicle statistics...")
                start_time = time.time()
                veh_df = vehicle_stats(vehicle_tour, service_events)
                veh_df_van = veh_df[veh_df.veh_class == "van"]
                veh_df_truck = veh_df[~(veh_df.veh_class == "van")]
                print(f"[{datetime.datetime.now()}] [INFO] Statistics calculated for {len(veh_df)} vehicles (Vans: {len(veh_df_van)}, Trucks: {len(veh_df_truck)}). (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [5/10] Processing vehicle data...")
                start_time = time.time()
                veh_df = process_vehicle_data(veh_df)
                print(f"[{datetime.datetime.now()}] [INFO] Vehicle data processed. (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [6/10] Extracting vehicle information...")
                start_time = time.time()
                vehicles = get_vehicles(vehicle_tour)
                print(f"[{datetime.datetime.now()}] [INFO] Extracted data for {len(vehicles)} vehicles. (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [7/10] Creating plot data...")
                start_time = time.time()
                plot_data, startG, endG = create_plot_data(vehicles, event_file)
                print(f"[{datetime.datetime.now()}] [INFO] Plot data created. (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [8/10] Integrating plot data with vehicle data...")
                start_time = time.time()
                result = plot_data.merge(veh_df, left_on='vehicle_id', right_on='vehicle_id')
                print(f"[{datetime.datetime.now()}] [INFO] Data integrated. (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [9/10] Parsing carriers from XML at {carrier_file}...")
                start_time = time.time()
                with gzip.open(carrier_file, mode="rt") as f:
                    tree = ET.parse(f)
                    root = tree.getroot()
                carriers = parse_carriers_from_xml(root)
                print(f"[{datetime.datetime.now()}] [INFO] Carriers parsed. (Runtime: {format_runtime(time.time() - start_time)})")

                print(f"[{datetime.datetime.now()}] [10/10] Augmenting result data with vehicle demand information...")
                start_time = time.time()
                result = add_vehicle_demand_to_result(carriers, result)

                # NEW: attach per-service start/end timestamps from MATSim events to each Service object
                print(f"[{datetime.datetime.now()}] [POST] Attaching service timestamps from events to Service objects...")
                start_time = time.time()
                result = attach_service_times_from_events(result, startG, endG)
                print(f"[{datetime.datetime.now()}] [INFO] Service timestamps attached. (Runtime: {format_runtime(time.time() - start_time)})")

                

                # def inspect_vehicle_stops(result_df, vehicle_id):
                #     """
                #     Print all stops of a vehicle with start, end, post_start_ts and durations.
                #     """
                #     row = result_df[result_df["vehicle_id"] == vehicle_id]
                #     if row.empty:
                #         print(f"No vehicle {vehicle_id} found.")
                #         return
                    
                #     services = row.iloc[0]["services"]
                #     print(f"Vehicle {vehicle_id}: {len(services)} services")
                #     for i, svc in enumerate(services, 1):
                #         st = svc.getAttribute("start_ts", None)
                #         et = svc.getAttribute("end_ts", None)
                #         pt = svc.getAttribute("post_start_ts", None)
                #         try:
                #             dur_decl = float(svc.getDuration())
                #         except Exception:
                #             dur_decl = None
                #         try:
                #             obs = (pd.to_datetime(et) - pd.to_datetime(st)).total_seconds() if st and et else None
                #         except Exception:
                #             obs = None

                #         print(f"Stop {i}:")
                #         print(f"  start_ts      = {st}")
                #         print(f"  end_ts        = {et}")
                #         print(f"  post_start_ts = {pt}")
                #         print(f"  declared_dur  = {dur_decl} s")
                #         print(f"  observed_dur  = {obs} s")
                #         print("-" * 40)

                # # Beispiel: erstes Fahrzeug in result inspizieren
                # first_vehicle_id = result["vehicle_id"].iloc[0]
                # inspect_vehicle_stops(result, first_vehicle_id)

                print(f"[{datetime.datetime.now()}] [POST] Processing Vehicle Tours...")
                vehicles_list = result['vehicle_id'].unique()
                time_columns = [f"{h:02}:{m:02}" for h in range(24) for m in range(60)]  # Generate time columns for every minute
                print(f"[{datetime.datetime.now()}] [POST] Generating Status DF... ")
                vehicle_status_df = pd.DataFrame(-3, index=vehicles_list, columns=time_columns)
                
                print(f"[{datetime.datetime.now()}] [POST] Processing Start:s ", len(startG))

                for name, group_start in tqdm(startG, desc="Processing vehicle tours", unit="tour"):
                    
                #     if "supply" in name:
                #         continue
                    
                    num_services = result[result.vehicle_id == name]["service_num"].iloc[0]
                    num_deliveries = result[result.vehicle_id == name]["deliveries"].iloc[0]
                    
                    tour_duration = result[result.vehicle_id == name]["Tour Duration"].iloc[0]
                    
                    services = result[result.vehicle_id == name]["services"].iloc[0]    

                    # Assuming that "Start time formatted" is already in a Timestamp format. If not, convert it.
                    start_time_tour = pd.to_datetime(result[result.vehicle_id == name]["Start time formatted"].iloc[0])
                    end_time_tour = pd.to_datetime(result[result.vehicle_id == name]["End time formatted"].iloc[0])
                    
                    loading_start = start_time_tour - pd.Timedelta(minutes=60)
                    
                    loading_start_minute = loading_start.round('min')
                    start_time_tour_minute = start_time_tour.round('min')
                    end_time_tour_minute = end_time_tour.round('min')
                    
                        # Check if loading_start is before start_time_tour
                    if loading_start_minute >= start_time_tour_minute:
                        raise Exception(f"For vehicle {name}, loading start time {loading_start_minute} is not before start time {start_time_tour_minute}.")

                    # Check if start_time_tour is before end_time_tour
                    if start_time_tour_minute >= end_time_tour_minute:
                        raise Exception(f"For vehicle {name}, start time {start_time_tour_minute} is not before end time {end_time_tour_minute}.")

                    # Check if loading_start is approximately 30 minutes before start_time_tour
                    time_difference = (start_time_tour_minute - loading_start_minute).seconds / 60
                    if not (29 <= time_difference <= 61):  # Allowing 1 minute buffer due to rounding
                        raise Exception(f"For vehicle {name}, the difference between loading start {loading_start_minute} and start time {start_time_tour_minute} is not approximately 30 minutes. Actual difference: {time_difference} minutes.")

                    
                    # Convert the rounded timestamps to HH:MM string format   
                    loading_start_str = loading_start_minute.strftime('%H:%M')    
                    loading_end_str = (start_time_tour_minute - pd.Timedelta(minutes=1)).strftime('%H:%M')    
                    
                    start_time_tour_str = start_time_tour_minute.strftime('%H:%M')
                    end_time_tour_str = end_time_tour_minute.strftime('%H:%M')

                    # Check if the times exist as columns in vehicle_status_df
                    if (loading_start_str in vehicle_status_df.columns) and (start_time_tour_str in vehicle_status_df.columns):
                        vehicle_status_df.loc[name, loading_start_str:start_time_tour_str] = -2  # Loading        

                    if (start_time_tour_str in vehicle_status_df.columns) and (end_time_tour_str in vehicle_status_df.columns):
                        vehicle_status_df.loc[name, start_time_tour_str:end_time_tour_str] = -1  # Driving
                    
                    veh_size = result[result.vehicle_id == name]["veh_size"].iloc[0]

                    # Set expected services based on vehicle size
                    if veh_size == "l":
                        expected_services = 230
                    elif veh_size == "m":
                        expected_services = 165
                    elif  veh_size == "supply_van":
                        expected_services = 230
                    elif veh_size == "truck_light":
                        expected_services = 1000        
                    elif veh_size == "truck":
                        expected_services = 2000
                    else:
                        raise Exception(f"Unexpected vehicle size: {veh_size}")

                    if num_services != (len(group_start) ):
                        raise Exception("Number of services does not match the expected value.")
                        
                    # Fetch and display corresponding group from endG
                    group_end = endG.get_group(name)
                    
                    # Concatenate the dataframes
                    combined_df = pd.concat([group_start, group_end])

                    # Sort by the 'time' column
                    sorted_df = combined_df.sort_values(by='time')
                    
                    startUtilization = (num_deliveries / expected_services) * 100    
                    
                        # Creating a new 'group_id' column where every two rows have the same ID
                    sorted_df['group_id'] = [i//2 for i, _ in enumerate(sorted_df.index)]

                    # Now, you can group by 'group_id' to get pairs of rows
                    grouped = [group for _, group in sorted_df.groupby('group_id')]
                    
                    current_load = num_deliveries

                    for index, g in enumerate(grouped):
                        correspondingService = services[index]

                        if len(g["link"].unique()) > 1:
                            raise Exception("Number of links for group does not match the expected value.")
                        if correspondingService.getServiceLink() != g["link"].unique()[0]:
                            raise Exception("Both Links are not matching for found Services", correspondingService.getServiceLink(), g["link"].unique()[0])

                        currentDemand = correspondingService.getDemand()

                        start = pd.to_datetime(g.iloc[0]["time"], unit='s').round('min')
                        end = pd.to_datetime(g.iloc[1]["time"], unit='s').round('min')

                        # Convert start and end to HH:MM format
                        start_str = start.strftime('%H:%M')
                        end_str = end.strftime('%H:%M')

                        # Calculate and round load percentage for the current state of the vehicle
                        load_percentage = round((current_load / expected_services) * 100)

                        # Update vehicle_status_df with the load percentage during the service activity
                        if (start_str in vehicle_status_df.columns) and (end_str in vehicle_status_df.columns):
                            vehicle_status_df.loc[name, start_str:end_str] = load_percentage 

                        # Reduce the current load by the deliveries made in this service activity
                        current_load -= currentDemand

                vans = result[result.veh_class == "van"]
                supplyTrucks = result[result['veh_class'].isin(['truck', 'truck_light'])]

                network_volumes_filtered = network_volumes[network_volumes.total_count > 0]

                # braucht: event_file (Pfad zur MATSim events.xml.gz) und network (GDF mit link_id/geometry)

                # def build_service_timeline(event_file, network=None):
                #     # Events lesen (actstart/actend für service)
                #     events = matsim.event_reader(event_file, types="actstart,actend")
                #     start_events, end_events = [], []
                #     for ev in events:
                #         if "_supply_" in ev.get("person",""):
                #             continue
                #         if ev.get("actType") == "service" and ev.get("type") == "actend":
                #             start_events.append(ev)   # service beginnt
                #         elif ev.get("actType") == "service" and ev.get("type") == "actstart":
                #             end_events.append(ev)     # service endet

                #     df_start = pd.DataFrame(start_events).sort_values(["person","time"]).reset_index(drop=True)
                #     df_end   = pd.DataFrame(end_events).sort_values(["person","time"]).reset_index(drop=True)
                #     if df_start.empty or df_end.empty:
                #         raise ValueError("Keine service start/end events gefunden.")

                #     geom_lookup = None
                #     if network is not None and {"link_id","geometry"}.issubset(network.columns):
                #         geom_lookup = network.set_index("link_id")["geometry"].to_dict()

                #     timelines_by_vehicle, frames = {}, []
                #     for person, st_grp in df_start.groupby("person", sort=False):
                #         en_grp = df_end[df_end["person"] == person].copy()
                #         if en_grp.empty:
                #             continue
                #         st_grp = st_grp.reset_index(drop=True)
                #         en_grp = en_grp.reset_index(drop=True)
                #         n = min(len(st_grp), len(en_grp))
                #         st, en = st_grp.iloc[:n], en_grp.iloc[:n]

                #         out = pd.DataFrame({
                #             "vehicle_id": person,
                #             "stop_index": range(n),
                #             "link_id": st["link"].values,
                #             "start_time_s": st["time"].astype(float).values,
                #             "end_time_s":   en["time"].astype(float).values
                #         })
                #         out["duration_s"] = (out["end_time_s"] - out["start_time_s"]).clip(lower=0)
                #         out["start_time"] = pd.to_datetime(out["start_time_s"], unit="s").dt.strftime("%H:%M:%S")
                #         out["end_time"]   = pd.to_datetime(out["end_time_s"],   unit="s").dt.strftime("%H:%M:%S")

                #         if geom_lookup is not None:
                #             def get_centroid_xy(lid):
                #                 g = geom_lookup.get(lid)
                #                 if g is None:
                #                     return pd.Series([None, None, None], index=["link_geom","x","y"])
                #                 c = g.centroid
                #                 return pd.Series([g, c.x, c.y], index=["link_geom","x","y"])
                #             out[["link_geom","x","y"]] = out["link_id"].apply(get_centroid_xy)

                #         timelines_by_vehicle[person] = out
                #         frames.append(out)

                #     timeline_all = pd.concat(frames, ignore_index=True).sort_values(
                #         ["vehicle_id","start_time_s"]
                #     ).reset_index(drop=True)
                #     return timelines_by_vehicle, timeline_all

                # # Aufruf:
                # timelines_by_vehicle, timeline_all = build_service_timeline(event_file, network=network)


                
                # === Filter low-utilization vehicles and adjust network volumes ===              
                THRESH_UTIL = 0.05  # threshold in [0..1] or [0..100] auto-detected
                print(f"[{datetime.datetime.now()}] [POST] Filter blow trashhold: ", THRESH_UTIL)

                low_util_ids = set()
                util_col = None

                if 'result' in globals() and isinstance(result, pd.DataFrame) and not result.empty:
                    # pick utilization column
                    if 'vehicle_load_factor' in result.columns:
                        util_col = 'vehicle_load_factor'
                    elif 'Vehicle Utilization (%)' in result.columns:
                        util_col = 'Vehicle Utilization (%)'
                    else:
                        print('[INFO] No utilization column found in result. Skipping low-util filter.')

                    if util_col is not None:
                        # detect scale (0..1 vs 0..100)
                        col_max = pd.to_numeric(result[util_col], errors='coerce').max()
                        thr = 100*THRESH_UTIL if (col_max is not None and col_max > 1.5) else THRESH_UTIL
                        # compute low-util vehicle ids
                        low_util_mask = pd.to_numeric(result[util_col], errors='coerce') < thr
                        low_util_ids = set(result.loc[low_util_mask, 'vehicle_id'].dropna().astype(str).unique())
                        keep_ids = set(result.loc[~low_util_mask, 'vehicle_id'].dropna().astype(str).unique())

                        # filtered result copy
                        result_filtered = result.loc[~low_util_mask].copy()
                        print(f"Low-util vehicles detected: {len(low_util_ids)} | kept: {len(keep_ids)}")

                        # filter tour dicts if present (for downstream use); dec_counts will be built from low-util ids
                        if 'vehicle_tour' in globals() and isinstance(vehicle_tour, dict):
                            vehicle_tour_filtered = {vid: lst for vid, lst in vehicle_tour.items() if str(vid) in keep_ids}

                        # adjust network volumes by subtracting traversals from low-util vehicles
                        if 'network_volumes_filtered' in globals() and isinstance(network_volumes_filtered, pd.DataFrame) and not network_volumes_filtered.empty:
                            nvf = network_volumes_filtered.copy()
                            link_col = 'link_id' if 'link_id' in nvf.columns else None
                            # choose count column to adjust
                            if 'van_count' in nvf.columns:
                                count_col = 'van_count'
                            elif 'veh_count' in nvf.columns:
                                count_col = 'veh_count'
                            elif 'count' in nvf.columns:
                                count_col = 'count'
                            else:
                                count_col = None

                            if link_col is None or count_col is None:
                                print('[WARN] Missing link_id or count column in network_volumes. Skipping adjustment.')
                            else:
                                # build decrement counts from tours for low-util vehicles
                                dec_counts = Counter()

                                # prefer original tours (include low-util), fallback to filtered only if original missing
                                vt_source = vehicle_tour if ('vehicle_tour' in globals() and isinstance(vehicle_tour, dict)) else None
                                if vt_source is None and 'vehicle_tour_filtered' in globals() and isinstance(vehicle_tour_filtered, dict):
                                    vt_source = vehicle_tour_filtered

                                def to_link_tokens(item):
                                    # items can be raw link ids or tuples like (link_id, time/idx)
                                    try:
                                        s = str(item[0]) if isinstance(item, tuple) and len(item) > 0 else str(item)
                                    except Exception:
                                        s = str(item)
                                    # split composite paths like "1210390-1210388" into individual link ids
                                    parts = [p.strip() for p in s.split('-') if str(p).strip() != '']
                                    return parts if parts else []

                                if vt_source is not None:
                                    for vid in low_util_ids:
                                        seq = vt_source.get(vid)
                                        if isinstance(seq, (list, tuple)):
                                            tokens = []
                                            for it in seq:
                                                tokens.extend(to_link_tokens(it))
                                            if tokens:
                                                dec_counts.update(tokens)

                                # map and subtract
                                if dec_counts:
                                    # normalize link ids for matching
                                    nvf_ids = nvf[link_col].astype(str).str.strip()
                                    dec_map = pd.Series(dec_counts)
                                    dec_series = nvf_ids.map(dec_map).fillna(0).astype(int)
                                    before_sum = pd.to_numeric(nvf[count_col], errors='coerce').fillna(0).sum()
                                    nvf[count_col] = (pd.to_numeric(nvf[count_col], errors='coerce').fillna(0) - dec_series).clip(lower=0).astype(int)
                                    after_sum = int(nvf[count_col].sum())
                                    matched = int((dec_series > 0).sum())
                                    network_volumes_filtered = nvf
                                    print(f"Adjusted network volumes: {int(before_sum)} -> {after_sum} ({count_col}) on {matched} links")
                                else:
                                    network_volumes_filtered = nvf
                                    print('[INFO] No decrement counts (no low-util tours matching). network_volumes_filtered = copy.')
                        else:
                            print('[INFO] network_volumes not present or empty; skipping network adjustment.')
                else:
                    print('[INFO] result not present; skipping low-util filter.')
                 
                print(f"[{datetime.datetime.now()}] [POST] Run EV Model")   

                # ============================================================
                # EV configuration and helpers
                # ============================================================

                # Global EV target across entire sample
                GLOBAL_EV_TARGET = 0.21

                # Fixed shares where we have strong evidence
                FIXED_EV_SHARES = {
                    "dhl": 0.4807,
                    "dpd": 0.0347,
                    "hermes": 0.1117,
                    "fedex": 0.0
                }

                # Minimal shares so flexible providers do not end at zero
                MIN_EV_SHARES = {
                    "amazon": 0.10,
                    "gls": 0.02,
                    "ups": 0.02
                }

                # Providers considered flexible after fixed shares are applied
                FLEXIBLE_PROVIDERS = {"amazon", "gls", "ups"}

                # EV assignment only for vans in this model
                EV_APPLY_TO_VANS_ONLY = True

                # EV effective classes used for calculation
                EV_CLASSES = {"van_e [< 2t]", "van_e [> 2t]"}

                # Electricity use for EV vans [kWh per km] by segment
                EV_ELEC_kWh_per_km = {
                    "van_e [< 2t]": {"urban": 0.25, "rural": 0.18, "highway": 0.22},
                    "van_e [> 2t]": {"urban": 0.30, "rural": 0.22, "highway": 0.27}
                }

                # WTW emission factor for electricity [g per kWh]
                # Set to 0.0 for market based green electricity with GoO
                # Use location based mix if required, for DE ~ 350 g/kWh
                EV_WTW_g_per_kWh = 0.0

                # Column used to rank "shortest tours first". Fallback to duration if missing.
                PRIMARY_DISTANCE_COL = "Tour_km"
                FALLBACK_DISTANCE_COL = "Tour Duration"

                # ---------- helper functions ----------

                def _extract_provider_from_vehicle_id(vid: str) -> str:
                    try:
                        return str(vid).split("_")[1].lower()
                    except Exception:
                        return "unknown"

                def _is_row_van(row) -> bool:
                    vs = str(row.get("veh_size", "")).lower()
                    return vs in {"m", "l", "supply_light_van"}

                def _veh_size_to_base_class(size_code: str) -> str | None:
                    return vehicle_size_mapping.get(str(size_code), None)

                def _ev_class_from_base(base_class: str) -> str:
                    if base_class == "van [< 2t]":
                        return "van_e [< 2t]"
                    if base_class == "van [> 2t]":
                        return "van_e [> 2t]"
                    return base_class

                def _distance_series_for_sort(df: pd.DataFrame) -> pd.Series:
                    if PRIMARY_DISTANCE_COL in df.columns:
                        return df[PRIMARY_DISTANCE_COL].astype(float)
                    if FALLBACK_DISTANCE_COL in df.columns:
                        return df[FALLBACK_DISTANCE_COL].astype(float)
                    return pd.Series(0.0, index=df.index)

                def compute_ev_counts_vans_only(df: pd.DataFrame) -> pd.DataFrame:
                    """
                    Compute EV counts per provider with exact match to the global target:
                    1. Apply fixed shares on total provider size, cap by vans only if enabled.
                    2. If fixed block overshoots the global target, downscale fixed providers proportionally.
                    3. If still remaining, satisfy minimal shares for flexible providers.
                    4. Distribute any rest to flexible providers by free van capacity.
                    5. Cap by van pool and keep counts non negative.
                    6. Final exactness pass to ensure sum equals the global target after caps.

                    Expects the following globals to exist:
                        GLOBAL_EV_TARGET: float in [0, 1]
                        FIXED_EV_SHARES: dict[str, float]
                        FLEXIBLE_PROVIDERS: set[str]
                        MIN_EV_SHARES: dict[str, float]
                        EV_APPLY_TO_VANS_ONLY: bool

                    Returns a DataFrame with columns:
                        provider, n_vehicles, n_vans, n_ev_target
                    """
                    tmp = df.copy()

                    # Provider tag
                    tmp["provider"] = tmp["vehicle_id"].map(_extract_provider_from_vehicle_id)
                    # Van eligibility
                    tmp["is_van"] = tmp.apply(_is_row_van, axis=1)

                    # Base stats per provider
                    fleet = tmp.groupby("provider")["vehicle_id"].count().rename("n_vehicles").reset_index()
                    vans = tmp[tmp["is_van"]].groupby("provider")["vehicle_id"].count().rename("n_vans").reset_index()
                    stats = pd.merge(fleet, vans, on="provider", how="left").fillna({"n_vans": 0})

                    # Dtypes
                    stats["n_vehicles"] = stats["n_vehicles"].astype(int)
                    stats["n_vans"] = stats["n_vans"].astype(int)

                    total = int(stats["n_vehicles"].sum())
                    target_total_evs = int(round(GLOBAL_EV_TARGET * total))

                    # Shares and flags
                    stats["fixed_share"] = stats["provider"].map(lambda p: FIXED_EV_SHARES.get(p, np.nan))
                    stats["is_flexible"] = stats["provider"].isin(FLEXIBLE_PROVIDERS)
                    stats["n_ev_target"] = 0

                    assigned = 0

                    # 1 Fixed block
                    for idx, row in stats.iterrows():
                        f = row["fixed_share"]
                        if pd.notna(f):
                            n = int(row["n_vehicles"])
                            n_vans = int(row["n_vans"])
                            ne = int(round(float(f) * n))
                            if EV_APPLY_TO_VANS_ONLY:
                                ne = min(ne, n_vans)
                            ne = max(0, ne)
                            stats.at[idx, "n_ev_target"] = ne
                            assigned += ne

                    remaining = target_total_evs - assigned

                    # 2 Downscale if fixed block overshoots the global target
                    if remaining < 0:
                        excess = -int(remaining)
                        fixed_mask = stats["fixed_share"].notna() & (stats["n_ev_target"] > 0)
                        if fixed_mask.any():
                            sub = stats.loc[fixed_mask, ["n_ev_target", "n_vans"]].copy()
                            total_fixed = int(sub["n_ev_target"].sum())
                            if total_fixed > 0:
                                target_after = max(0, total_fixed - excess)
                                # proportional scale
                                scaled = sub["n_ev_target"].astype(float) * (float(target_after) / float(total_fixed))
                                floored = scaled.apply(np.floor).astype(int)  # aligned Series
                                rema = scaled - floored
                                need = int(target_after - int(floored.sum()))
                                if need > 0:
                                    for idx2 in rema.sort_values(ascending=False).index[:need]:
                                        floored.loc[idx2] += 1
                                # cap by van pool and zero
                                floored = floored.clip(lower=0)
                                floored = np.minimum(floored, sub["n_vans"])
                                # fix small residuals after caps
                                diff = int(target_after - int(floored.sum()))
                                if diff != 0:
                                    headroom = (sub["n_vans"] - floored).astype(int)
                                    if diff > 0:
                                        for idx2 in headroom.sort_values(ascending=False).index:
                                            if headroom.loc[idx2] <= 0:
                                                continue
                                            take = min(headroom.loc[idx2], diff)
                                            floored.loc[idx2] += take
                                            diff -= take
                                            if diff == 0:
                                                break
                                    else:
                                        for idx2 in floored.sort_values(ascending=False).index:
                                            if floored.loc[idx2] <= 0:
                                                continue
                                            take = min(floored.loc[idx2], -diff)
                                            floored.loc[idx2] -= take
                                            diff += take
                                            if diff == 0:
                                                break
                                # write back
                                stats.loc[floored.index, "n_ev_target"] = floored.astype(int)
                        assigned = int(stats["n_ev_target"].sum())
                        remaining = target_total_evs - assigned

                    # 3 Minimal shares for flexible providers
                    if remaining > 0:
                        flex_view = stats[stats["is_flexible"]]
                        for idx, row in flex_view.iterrows():
                            p = row["provider"]
                            n = int(row["n_vehicles"])
                            n_vans = int(row["n_vans"])
                            mshare = float(MIN_EV_SHARES.get(p, 0.0))
                            need = int(round(mshare * n))
                            if EV_APPLY_TO_VANS_ONLY:
                                need = min(need, n_vans)
                            take = min(max(0, need), max(0, remaining))
                            if take > 0:
                                stats.at[idx, "n_ev_target"] += take
                                remaining -= take
                                if remaining <= 0:
                                    break

                    # 4 Distribute rest by free van capacity across flexible providers
                    if remaining > 0 and stats["is_flexible"].any():
                        flex = stats[stats["is_flexible"]].copy()
                        flex["free"] = (flex["n_vans"] - flex["n_ev_target"]).clip(lower=0).astype(int)
                        cap = int(flex["free"].sum())
                        if cap > 0:
                            # proportional split
                            for idx, row in flex.iterrows():
                                free = int(row["free"])
                                if free <= 0:
                                    continue
                                add = int(round(remaining * (free / cap)))
                                add = min(add, free)
                                if add > 0:
                                    stats.at[idx, "n_ev_target"] += add
                            # rounding fix
                            diff = target_total_evs - int(stats["n_ev_target"].sum())
                            if diff != 0:
                                for idx in stats[stats["is_flexible"]].index:
                                    free = int(stats.at[idx, "n_vans"] - stats.at[idx, "n_ev_target"])
                                    if diff > 0 and free > 0:
                                        stats.at[idx, "n_ev_target"] += 1
                                        diff -= 1
                                    elif diff < 0 and stats.at[idx, "n_ev_target"] > 0:
                                        stats.at[idx, "n_ev_target"] -= 1
                                        diff += 1
                                    if diff == 0:
                                        break

                    # 5 Final cap by van pool and non negativity
                    stats["n_ev_target"] = stats.apply(
                        lambda r: int(max(0, min(int(r["n_ev_target"]), int(r["n_vans"])))),
                        axis=1
                    )

                    # 6 Final exactness pass to match the global target after caps
                    total_assigned_plan = int(stats["n_ev_target"].sum())
                    if total_assigned_plan != target_total_evs:
                        diff = target_total_evs - total_assigned_plan
                        # prefer flexible providers for adjustments
                        preferred = list(stats.index[stats["is_flexible"]]) + list(stats.index)
                        if diff > 0:
                            # add where headroom exists
                            for idx in preferred:
                                head = int(stats.at[idx, "n_vans"] - stats.at[idx, "n_ev_target"])
                                if head <= 0:
                                    continue
                                take = min(head, diff)
                                stats.at[idx, "n_ev_target"] += take
                                diff -= take
                                if diff == 0:
                                    break
                        elif diff < 0:
                            # remove where assigned is positive
                            for idx in preferred:
                                cur = int(stats.at[idx, "n_ev_target"])
                                if cur <= 0:
                                    continue
                                take = min(cur, -diff)
                                stats.at[idx, "n_ev_target"] -= take
                                diff += take
                                if diff == 0:
                                    break

                        # Validate again
                        total_assigned_plan = int(stats["n_ev_target"].sum())
                        if total_assigned_plan != target_total_evs:
                            raise RuntimeError(
                                f"EV planning mismatch after final cap, have {total_assigned_plan} but target is {target_total_evs}"
                            )

                    return stats[["provider", "n_vehicles", "n_vans", "n_ev_target"]]
                # def compute_ev_counts_vans_only(df: pd.DataFrame) -> pd.DataFrame:
                #     """
                #     Compute EV counts per provider:
                #     1 fixed shares applied on total provider size but capped by vans only
                #     2 minimal shares for flexible providers
                #     3 distribute remaining to flexible providers by free van capacity
                #     4 cap by van pool and keep counts non negative
                #     Returns columns: provider, n_vehicles, n_vans, n_ev_target
                #     """
                #     tmp = df.copy()
                #     # Provider ableiten
                #     tmp["provider"] = tmp["vehicle_id"].map(_extract_provider_from_vehicle_id)
                #     # Van-Flag
                #     tmp["is_van"] = tmp.apply(_is_row_van, axis=1)

                #     # Basisstatistiken
                #     fleet = tmp.groupby("provider")["vehicle_id"].count().rename("n_vehicles").reset_index()
                #     vans = tmp[tmp["is_van"]].groupby("provider")["vehicle_id"].count().rename("n_vans").reset_index()
                #     stats = pd.merge(fleet, vans, on="provider", how="left").fillna({"n_vans": 0})

                #     # Datentypen sichern
                #     stats["n_vehicles"] = stats["n_vehicles"].astype(int)
                #     stats["n_vans"] = stats["n_vans"].astype(int)

                #     total = int(stats["n_vehicles"].sum())
                #     target_total_evs = int(round(GLOBAL_EV_TARGET * total))

                #     # Shares mappen
                #     stats["fixed_share"] = stats["provider"].map(lambda p: FIXED_EV_SHARES.get(p, np.nan))
                #     stats["is_flexible"] = stats["provider"].isin(FLEXIBLE_PROVIDERS)
                #     stats["n_ev_target"] = 0

                #     assigned = 0

                #     # 1) Fixed block, nur wenn fixed_share eine Zahl ist
                #     for idx, row in stats.iterrows():
                #         f = row["fixed_share"]
                #         if pd.notna(f):
                #             n = int(row["n_vehicles"])
                #             n_vans = int(row["n_vans"])
                #             ne = int(round(float(f) * n))
                #             if EV_APPLY_TO_VANS_ONLY:
                #                 ne = min(ne, n_vans)
                #             ne = max(0, ne)
                #             stats.at[idx, "n_ev_target"] = ne
                #             assigned += ne

                #     remaining = target_total_evs - assigned

                #     # 2) Minimalanteile für flexible Provider
                #     if remaining > 0:
                #         for idx, row in stats[stats["is_flexible"]].iterrows():
                #             p = row["provider"]
                #             n = int(row["n_vehicles"])
                #             n_vans = int(row["n_vans"])
                #             mshare = float(MIN_EV_SHARES.get(p, 0.0))
                #             need = int(round(mshare * n))
                #             if EV_APPLY_TO_VANS_ONLY:
                #                 need = min(need, n_vans)
                #             take = min(max(0, need), max(0, remaining))
                #             stats.at[idx, "n_ev_target"] += take
                #             remaining -= take
                #             if remaining <= 0:
                #                 break

                #     # 3) Rest proportional nach freier Van-Kapazität auf flexible Provider
                #     if remaining > 0 and stats["is_flexible"].any():
                #         flex = stats[stats["is_flexible"]].copy()
                #         flex["free"] = (flex["n_vans"] - flex["n_ev_target"]).clip(lower=0).astype(int)
                #         cap = int(flex["free"].sum())
                #         if cap > 0:
                #             for idx, row in flex.iterrows():
                #                 free = int(row["free"])
                #                 if free <= 0:
                #                     continue
                #                 add = int(round(remaining * (free / cap)))
                #                 add = min(add, free)
                #                 stats.at[idx, "n_ev_target"] += add

                #             # Rounding-Korrektur
                #             diff = target_total_evs - int(stats["n_ev_target"].sum())
                #             if diff != 0:
                #                 for idx in stats[stats["is_flexible"]].index:
                #                     free = int(stats.at[idx, "n_vans"] - stats.at[idx, "n_ev_target"])
                #                     if diff > 0 and free > 0:
                #                         stats.at[idx, "n_ev_target"] += 1
                #                         diff -= 1
                #                     elif diff < 0 and stats.at[idx, "n_ev_target"] > 0:
                #                         stats.at[idx, "n_ev_target"] -= 1
                #                         diff += 1
                #                     if diff == 0:
                #                         break

                #     # 4) Finaler Cap und Non-Negativität
                #     stats["n_ev_target"] = stats.apply(
                #         lambda r: int(max(0, min(int(r["n_ev_target"]), int(r["n_vans"])))),
                #         axis=1
                #     )

                #     return stats[["provider", "n_vehicles", "n_vans", "n_ev_target"]]

                def assign_ev_flags_shortest_first(df: pd.DataFrame, ev_counts: pd.DataFrame) -> pd.DataFrame:
                    """
                    Adds 'is_ev' to df. Only vans eligible. Within each provider
                    the vehicles with shortest distance become EV.
                    """
                    out = df.copy()
                    out["provider"] = out["vehicle_id"].map(_extract_provider_from_vehicle_id)
                    out["is_van"] = out.apply(_is_row_van, axis=1)
                    out["is_ev"] = 0
                    dists = _distance_series_for_sort(out)
                    need = {r["provider"]: int(r["n_ev_target"]) for _, r in ev_counts.iterrows()}

                    for prov, sub in out.groupby("provider"):
                        n_ev = int(need.get(prov, 0))
                        if n_ev <= 0:
                            continue
                        cand = sub[sub["is_van"]].copy()
                        if cand.empty:
                            continue
                        order = cand.index.values[np.argsort(dists.loc[cand.index].to_numpy())]
                        take = order[:n_ev]
                        out.loc[take, "is_ev"] = 1
                    return out

                def build_effective_class_map(df_with_flags: pd.DataFrame) -> dict[str, str]:
                    """
                    Map vehicle_id to effective calculation class. EV vans become EV classes.
                    Others keep their base class.
                    """
                    eff = {}
                    for _, row in df_with_flags.iterrows():
                        vid = row["vehicle_id"]
                        base = _veh_size_to_base_class(row.get("veh_size", ""))
                        if base is None:
                            continue
                        eff[vid] = _ev_class_from_base(base) if int(row.get("is_ev", 0)) == 1 and base in {"van [< 2t]", "van [> 2t]"} else base
                    return eff

                def ev_sanity_report(result_df: pd.DataFrame, ev_plan: pd.DataFrame, global_target: float = GLOBAL_EV_TARGET):
                    """
                    Print a detailed EV assignment sanity report by provider and overall.
                    Resolves overlapping columns and tolerates missing 'provider' in result_df.
                    """
                    df = result_df.copy()

                    # Ensure provider column exists in result
                    if "provider" not in df.columns:
                        df["provider"] = df["vehicle_id"].map(_extract_provider_from_vehicle_id)

                    # Aggregate assigned EVs and count of vehicles from result
                    assigned = (df.groupby("provider")["is_ev"]
                                .agg(assigned_evs="sum", n_from_result="count")
                                .reset_index())

                    # ev_plan expected columns: provider, n_vehicles, n_vans, n_ev_target
                    plan = ev_plan.copy()
                    for col in ["provider", "n_vehicles", "n_vans", "n_ev_target"]:
                        if col not in plan.columns:
                            raise ValueError(f"[EV] ev_plan is missing required column '{col}'")

                    # Merge without column collision: keep 'n_vehicles' from ev_plan, call result count 'n_from_result'
                    merged = plan.merge(assigned, on="provider", how="outer")

                    # Fill NaNs to zeros for numeric columns
                    num_cols = ["n_vehicles", "n_vans", "n_ev_target", "assigned_evs", "n_from_result"]
                    for col in num_cols:
                        if col in merged.columns:
                            merged[col] = merged[col].fillna(0).astype(int)

                    # Choose denominator per provider:
                    # Prefer ev_plan['n_vehicles'] if present, else fallback to n_from_result
                    den = merged["n_vehicles"].where(merged["n_vehicles"] > 0, merged["n_from_result"])
                    den = den.replace(0, np.nan)  # avoid div by zero

                    merged["target_share_%"] = np.where(den.notna(), merged["n_ev_target"] / den * 100.0, 0.0)
                    merged["assigned_share_%"] = np.where(den.notna(), merged["assigned_evs"] / den * 100.0, 0.0)

                    # Totals
                    total_from_plan = int(merged["n_vehicles"].sum())
                    total_from_result = int(merged["n_from_result"].sum())
                    total_vehicles = total_from_plan if total_from_plan > 0 else total_from_result

                    assigned_total = int(merged["assigned_evs"].sum())
                    target_total = int(round(global_target * total_vehicles))

                    # Header
                    print("\n[EV] Sanity report")
                    print("=" * 72)
                    print(f"Global target: {target_total} EVs ({global_target*100:.1f}% of {total_vehicles} vehicles)")
                    print(f"Assigned EVs: {assigned_total} ({(assigned_total/total_vehicles*100 if total_vehicles else 0):.1f}%)\n")

                    # Per provider pretty print, sorted by provider name
                    cols_for_print = ["provider", "n_vehicles", "n_vans", "n_ev_target", "target_share_%",
                                    "assigned_evs", "assigned_share_%"]
                    merged = merged[cols_for_print].sort_values("provider")

                    for _, r in merged.iterrows():
                        prov = str(r["provider"])
                        nveh = int(r["n_vehicles"])
                        nvans = int(r["n_vans"])
                        tgt = int(r["n_ev_target"])
                        tgt_pct = float(r["target_share_%"])
                        asg = int(r["assigned_evs"])
                        asg_pct = float(r["assigned_share_%"])
                        print(f"{prov:<10} | Vehicles: {nveh:4d} | Vans: {nvans:4d} | "
                            f"EV target: {tgt:4d} ({tgt_pct:5.1f}%) | "
                            f"EV assigned: {asg:4d} ({asg_pct:5.1f}%)")

                    print("=" * 72)
                    print(f"TOTAL     | Vehicles: {total_vehicles:4d} | Target: {target_total:4d} "
                        f"({global_target*100:.1f}%) | Assigned: {assigned_total:4d} "
                        f"({(assigned_total/total_vehicles*100 if total_vehicles else 0):.1f}%)")
                    print("=" * 72)


                # ============================================================
                # EV preparation before the main per vehicle loop
                # ============================================================

                # Create a mapping dictionary
                vehicle_size_mapping = {
                    'm': 'van [< 2t]',
                    'l': 'van [> 2t]',
                    'truck': 'truck [10-20 t] + trailer',
                    'truck_light': 'truck [< 10t]',
                    'supply_light_van': 'van [> 2t]',
                }

                # 1 compute EV targets on vans only
                _ev_plan = compute_ev_counts_vans_only(result)

                # 2 assign EV flags to concrete vehicles by shortest distance first
                result = assign_ev_flags_shortest_first(result, _ev_plan)

                # 3 build effective class map and fast lookups
                EFFECTIVE_CLASS_MAP = build_effective_class_map(result)
                IS_EV_MAP = result.set_index("vehicle_id")["is_ev"].to_dict()

                BASE_CLASS_MAP = {}
                for vid, row in result.set_index("vehicle_id")[["veh_size"]].itertuples():
                    base = _veh_size_to_base_class(row)
                    if base is not None:
                        BASE_CLASS_MAP[vid] = base

                # sanity report
                ev_sanity_report(result, _ev_plan, GLOBAL_EV_TARGET)

                def _effective_class_for_calc_from_map(veh_id: str) -> str | None:
                    return EFFECTIVE_CLASS_MAP.get(veh_id, None)

                def _base_bucket_class_from_map(veh_id: str) -> str | None:
                    return BASE_CLASS_MAP.get(veh_id, None)



                # ============================================================
                # Main loop with EV aware drive, idle, and cold start handling
                # Basis switch for "CO2" vs "CO2e"
                # ============================================================

                # -*- coding: utf-8 -*-
                # Pipeline can output TTW CO2 or TTW CO2e (with optional WTW add-on)
                # Includes optional idle local pollutants (NOx, HC, CO)


                # ============================================================
                # User controls
                # ============================================================

                WITHOUT_SUPPLY_TRUCKS = False     # True => only last-mile vans are counted
                T_LONG_OVERRIDE = None            # 0 => force engine off at every stop, None => use class thresholds
                SIM_DATE = pd.Timestamp("2025-05-13")

                # Choose emissions basis. Options: "CO2" or "CO2e"
                EMISSIONS_BASIS = "CO2"           # set to "CO2e" to include CH4 and N2O with GWP
                USE_WTW = False                   # only applies if EMISSIONS_BASIS == "CO2e"

                # Idle local pollutants accounting
                IDLE_POLLUTANTS_ON = True

                # Base idle emission rates in g/s for a reference heavy-duty diesel at warm idle (local pollutants)
                # These are only used for NOx, HC, CO, not for climate gases
                BASE_IDLE_CO_gps   = 0.005    # ~ 18 g/h
                BASE_IDLE_HC_gps   = 0.002    # ~ 7 g/h
                BASE_IDLE_NOX_gps  = 0.020    # ~ 72 g/h

                # Multipliers by vehicle class for idle pollutants (local pollutants only)
                IDLE_MULT_BY_CLASS = {
                    "van [< 2t]": 0.40,
                    "van [> 2t]": 0.60,
                    "truck [< 10t]": 1.00,
                    "truck [10-20 t] + trailer": 1.20
                }

                def idle_pollutant_temp_mult(temp_c: float) -> float:
                    """Mild temperature multiplier for idle local pollutants."""
                    if temp_c <= 0:
                        return 1.15
                    if temp_c >= 25:
                        return 1.05
                    return 1.00

                # ============================================================
                # Global constants and lookups
                # ============================================================

                # GWP reference set. AR4 by default for backward compatibility.
                # If you want AR6 fossil 100y use: {"CO2": 1.0, "CH4": 27.2, "N2O": 273.0}
                GWP = {"CO2": 1.0, "CH4": 25.0, "N2O": 298.0}

                MONTHLY_AVG_TEMP_C = {1:0.0, 2:1.0, 3:5.0, 4:9.0, 5:14.0, 6:17.0,
                                    7:19.0, 8:18.0, 9:15.0, 10:10.0, 11:5.0, 12:2.0}
                MONTH_MULT = {1:1.08, 2:1.08, 3:1.03, 4:1.03, 5:1.02, 6:1.02,
                            7:1.02, 8:1.02, 9:1.03, 10:1.03, 11:1.08, 12:1.08}

                AMBIENT_TEMP_C = MONTHLY_AVG_TEMP_C[SIM_DATE.month]
                SEASON_MULT    = MONTH_MULT[SIM_DATE.month]

                # STREAM CO2 factors (g per km), min..max for empty..full load
                # These are CO2 only. CH4 and N2O are provided separately below if CO2e is requested.
                EMISSION_CO2_gpkm = {
                    "van [< 2t]": {"urban": (197, 213), "rural": (117, 126), "highway": (171, 185)},
                    "van [> 2t]": {"urban": (276, 302), "rural": (170, 186), "highway": (250, 275)},
                    "truck [< 10t]": {"urban": (419, 472), "rural": (281, 316), "highway": (253, 286)},
                    "truck [10-20 t] + trailer": {"urban": (1023, 1387), "rural": (658, 892), "highway": (559, 757)}
                }

                # STREAM energy demand (MJ per km) for WTT uplift on vans; only used if EMISSIONS_BASIS == "CO2e" and USE_WTW True
                ENERGY_MJ_per_km = {
                    "van [< 2t]": {"urban": (2.7, 2.9), "rural": (1.6, 1.7), "highway": (2.3, 2.5)},
                    "van [> 2t]": {"urban": (3.7, 4.1), "rural": (2.3, 2.5), "highway": (3.3, 3.7)}
                }
                WTT_CO2e_g_per_MJ = 24.0

                # Vehicle class parameters for idle and cold start
                # Interpretations:
                # - idle_g_per_sec_ttw: TTW CO2 grams per second at engine on; we apply engine_on_share and seasonal multiplier
                # - cold_start_base_co2_g: TTW CO2 grams for a cold start event at the given class baseline
                VEH_CLASS_PARAMS = {
                    "van [< 2t]": {
                        "idle_g_per_sec_ttw": 0.80,
                        "t_short": 120, "t_mid": 240, "t_long": 600,
                        "curbside_mult": 1.10, "off_threshold": 0.30,
                        "cold_start_base_co2_g": 8.0, "min_cold_dwell": 300
                    },
                    "van [> 2t]": {
                        "idle_g_per_sec_ttw": 0.90,
                        "t_short": 120, "t_mid": 240, "t_long": 600,
                        "curbside_mult": 1.10, "off_threshold": 0.30,
                        "cold_start_base_co2_g": 10.0, "min_cold_dwell": 300
                    },
                    "truck [< 10t]": {
                        "idle_g_per_sec_ttw": 1.40,
                        "t_short": 180, "t_mid": 540, "t_long": 900,
                        "curbside_mult": 1.05, "off_threshold": 0.25,
                        "cold_start_base_co2_g": 12.0, "min_cold_dwell": 600
                    },
                    "truck [10-20 t] + trailer": {
                        "idle_g_per_sec_ttw": 1.60,
                        "t_short": 180, "t_mid": 540, "t_long": 900,
                        "curbside_mult": 1.05, "off_threshold": 0.25,
                        "cold_start_base_co2_g": 14.0, "min_cold_dwell": 600
                    }
                }
                DEFAULT_PARAMS = {
                    "idle_g_per_sec_ttw": 1.20,
                    "t_short": 120, "t_mid": 360, "t_long": 600,
                    "curbside_mult": 1.10, "off_threshold": 0.30,
                    "cold_start_base_co2_g": 10.0, "min_cold_dwell": 300
                }

                vehicle_size_mapping = {
                    "m": "van [< 2t]",
                    "l": "van [> 2t]",
                    "truck": "truck [10-20 t] + trailer",
                    "truck_light": "truck [< 10t]",
                    "supply_light_van": "van [> 2t]"
                }
                SUPPLY_TRUCK_CLASSES = {"truck [< 10t]", "truck [10-20 t] + trailer"}

                # ============================================================
                # CH4 and N2O defaults with min..max interpolation for CO2e
                # ============================================================

                CH4_LOAD_SCALE = 0.15
                N2O_LOAD_SCALE = 0.15
                def _span(v, scale): return (v * (1.0 - scale), v * (1.0 + scale))

                _CH4_BASE = {
                    "van [< 2t]": {"urban": 0.0045, "rural": 0.0035, "highway": 0.0035},
                    "van [> 2t]": {"urban": 0.0055, "rural": 0.0045, "highway": 0.0045},
                    "truck [< 10t]": {"urban": 0.0080, "rural": 0.0060, "highway": 0.0060},
                    "truck [10-20 t] + trailer": {"urban": 0.0100, "rural": 0.0080, "highway": 0.0080}
                }
                _N2O_BASE = {
                    "van [< 2t]": {"urban": 0.0190, "rural": 0.0140, "highway": 0.0160},
                    "van [> 2t]": {"urban": 0.0250, "rural": 0.0180, "highway": 0.0200},
                    "truck [< 10t]": {"urban": 0.0400, "rural": 0.0300, "highway": 0.0350},
                    "truck [10-20 t] + trailer": {"urban": 0.0600, "rural": 0.0450, "highway": 0.0500}
                }
                def _build_minmax_from_base(base_dict, scale):
                    out = {}
                    for vclass, segs in base_dict.items():
                        out[vclass] = {}
                        for seg, base in segs.items():
                            out[vclass][seg] = _span(base, scale)
                    return out

                EMISSION_CH4_gpkm_defaults = _build_minmax_from_base(_CH4_BASE, CH4_LOAD_SCALE)
                EMISSION_N2O_gpkm_defaults = _build_minmax_from_base(_N2O_BASE, N2O_LOAD_SCALE)

                if "EMISSION_CH4_gpkm" not in globals(): EMISSION_CH4_gpkm = {}
                if "EMISSION_N2O_gpkm" not in globals(): EMISSION_N2O_gpkm = {}
                def _merge_minmax(dst, src):
                    for vclass, segs in src.items():
                        if vclass not in dst: dst[vclass] = {}
                        for seg, pair in segs.items():
                            dst[vclass].setdefault(seg, pair)
                _merge_minmax(EMISSION_CH4_gpkm, EMISSION_CH4_gpkm_defaults)
                _merge_minmax(EMISSION_N2O_gpkm, EMISSION_N2O_gpkm_defaults)

                def convert_seconds_to_timestamp(seconds):
                    seconds = float(seconds)
                    time_timedelta = pd.to_timedelta(seconds, unit="s")
                    reference_time = pd.Timestamp("00:00:00")
                    return (reference_time + time_timedelta).round("min")

                # ============================================================
                # Helpers for factors and interpolation
                # ============================================================

                def _interp(min_val, max_val, load_pct):
                    lp = float(max(0.0, min(100.0, load_pct)))
                    return min_val + (max_val - min_val) * (lp / 100.0)

                def _get_factor(table, vehicle_type, segment_type, load_pct):
                    if vehicle_type not in table: return None
                    segs = table[vehicle_type]
                    if segment_type not in segs: return None
                    lo, hi = segs[segment_type]
                    return _interp(lo, hi, load_pct)

                def _species_to_co2e_g(co2_g: float, ch4_g: float, n2o_g: float) -> float:
                    """Convert species grams to CO2e grams using global GWP."""
                    return co2_g * GWP["CO2"] + ch4_g * GWP["CH4"] + n2o_g * GWP["N2O"]

                def ttw_gpkm(vehicle_type, segment_type, load_pct, basis="CO2"):
                    """Return TTW grams per km for the selected basis."""
                    co2 = _get_factor(EMISSION_CO2_gpkm, vehicle_type, segment_type, load_pct)
                    if co2 is None: 
                        return None
                    if basis == "CO2":
                        return co2
                    # CO2e branch
                    ch4 = _get_factor(EMISSION_CH4_gpkm, vehicle_type, segment_type, load_pct) or 0.0
                    n2o = _get_factor(EMISSION_N2O_gpkm, vehicle_type, segment_type, load_pct) or 0.0
                    return _species_to_co2e_g(co2, ch4, n2o)

                def calc_wtt_from_energy_g_per_km(vehicle_type, segment_type, load_pct):
                    """Return WTT grams CO2e per km for vans if energy intensity is available."""
                    mj = _get_factor(ENERGY_MJ_per_km, vehicle_type, segment_type, load_pct)
                    if mj is None: 
                        return None
                    return mj * WTT_CO2e_g_per_MJ

                def calc_emissions_drive(distance_m, vehicle_type, segment_type, load_percentage, basis=EMISSIONS_BASIS):
                    """Distance based emissions for the selected basis. Optional WTW uplift only if basis is CO2e."""
                    if WITHOUT_SUPPLY_TRUCKS and vehicle_type in SUPPLY_TRUCK_CLASSES:
                        return 0.0
                    distance_km = float(distance_m) / 1000.0
                    gpkm = ttw_gpkm(vehicle_type, segment_type, load_percentage, basis=basis)
                    if gpkm is None: 
                        return 0.0
                    total_gpkm = gpkm
                    if basis == "CO2e" and USE_WTW:
                        wtt = calc_wtt_from_energy_g_per_km(vehicle_type, segment_type, load_percentage)
                        if wtt is not None:
                            total_gpkm += wtt
                    return distance_km * total_gpkm

                # ============================================================
                # Idle and cold start emissions for basis selection
                # ============================================================

                def engine_on_share(dwell_sec, vehicle_type, ambient_temp_c, curbside=True, t_long_override=None):
                    """
                    Returns the fraction of dwell time with engine on.
                    Target profile tuned for low on-time at curbside dwells.
                    """
                    if WITHOUT_SUPPLY_TRUCKS and vehicle_type in SUPPLY_TRUCK_CLASSES:
                        return 0.0

                    p = VEH_CLASS_PARAMS.get(vehicle_type, DEFAULT_PARAMS)
                    s = float(max(0.0, dwell_sec))

                    if t_long_override == 0:
                        return 0.0

                    t_short = p["t_short"]
                    t_mid = p["t_mid"]
                    t_long = p["t_long"] if t_long_override is None else t_long_override

                    short_share = 0.10
                    mid_share   = 0.01
                    long_share  = 0.00

                    if s <= t_short:
                        share = short_share
                    elif s <= t_mid:
                        denom = max(1.0, (t_mid - t_short))
                        share = short_share + (mid_share - short_share) * ((s - t_short) / denom)
                    elif s <= t_long:
                        denom = max(1.0, (t_long - t_mid))
                        share = mid_share + (long_share - mid_share) * ((s - t_mid) / denom)
                    else:
                        share = 0.03

                    if curbside:
                        share *= p.get("curbside_mult", 1.0)
                    else:
                        share *= 0.85

                    if ambient_temp_c <= 0:
                        share *= 1.10
                    elif ambient_temp_c >= 25:
                        share *= 1.03

                    share = float(max(0.01, min(1.0, share)))
                    return share

                def idle_emissions(dwell_sec, vehicle_type, ambient_temp_c, curbside=True, t_long_override=None, basis=EMISSIONS_BASIS, is_ev=False):
                    """
                    TTW idle emissions for the selected basis.
                    Assumptions:
                    - idle_g_per_sec_ttw parameter is CO2 g per second at engine on.
                    - CH4 and N2O at idle are neglected. This keeps results comparable to studies that report CO2 only.
                    - If basis == CO2e we return the same grams as CO2, since CH4 and N2O at idle are negligible here.
                    - WTW uplift is not added at idle. If needed, add a factor similar to drive path.
                    """
                    if is_ev:
                        return 0.0
                    
                    if WITHOUT_SUPPLY_TRUCKS and vehicle_type in SUPPLY_TRUCK_CLASSES:
                        return 0.0
                    p = VEH_CLASS_PARAMS.get(vehicle_type, DEFAULT_PARAMS)
                    share = engine_on_share(
                        dwell_sec=dwell_sec,
                        vehicle_type=vehicle_type,
                        ambient_temp_c=ambient_temp_c,
                        curbside=curbside,
                        t_long_override=t_long_override
                    )
                    g_per_sec_eff = p["idle_g_per_sec_ttw"] * share * SEASON_MULT
                    return float(max(0.0, dwell_sec)) * g_per_sec_eff

                DEFAULT_MIN_COLD_DWELL = 300
                def effective_cold_dwell_threshold(vehSize, ambient_temp_c):
                    """Temperature aware minimum dwell threshold before a cold start can occur."""
                    p = VEH_CLASS_PARAMS.get(vehSize, {})
                    base_threshold = p.get("min_cold_dwell", DEFAULT_MIN_COLD_DWELL)
                    if ambient_temp_c < 0:
                        factor = 0.5
                    elif ambient_temp_c > 20:
                        factor = 1.5
                    else:
                        factor = 1.0
                    return base_threshold * factor

                def cold_start_emissions(vehicle_type, ambient_temp_c, basis=EMISSIONS_BASIS):
                    """
                    Cold start TTW grams for the selected basis.
                    Assumptions:
                    - cold_start_base_co2_g parameter is TTW CO2 grams per cold start event at baseline temperature.
                    - We apply a very mild temperature factor to CO2 only. CH4 and N2O are not added.
                    - If basis == CO2e we return same grams as CO2 for consistency with comparators that ignore species at cold start.
                    """
                    p = VEH_CLASS_PARAMS.get(vehicle_type, DEFAULT_PARAMS)
                    base = float(p.get("cold_start_base_co2_g", 0.0))
                    # Simple temperature effect. You can refine this if you have data.
                    if ambient_temp_c <= 0:
                        mult = 1.10
                    elif ambient_temp_c >= 25:
                        mult = 0.98
                    else:
                        mult = 1.00
                    return base * mult

                def motor_was_off(dwell_sec,
                                vehicle_type,
                                ambient_temp_c,
                                curbside=True,
                                t_long_override=None,
                                is_ev=False):
                    """
                    Decide if engine was off during the stop.
                    This flag is used only to allow a cold start on departure.
                    EV never causes a cold start.
                    """
                    # EV cannot have a tailpipe cold start
                    if is_ev:
                        return False

                    # Supply truck filter
                    if WITHOUT_SUPPLY_TRUCKS and vehicle_type in SUPPLY_TRUCK_CLASSES:
                        return False

                    p = VEH_CLASS_PARAMS.get(vehicle_type, DEFAULT_PARAMS)

                    # Determine share of engine on during dwell
                    share = engine_on_share(
                        dwell_sec=dwell_sec,
                        vehicle_type=vehicle_type,
                        ambient_temp_c=ambient_temp_c,
                        curbside=curbside,
                        t_long_override=t_long_override
                    )

                    # Long stop condition
                    t_long = p["t_long"] if t_long_override is None else t_long_override
                    long_stop = float(dwell_sec) >= float(t_long)

                    # If share falls below off_threshold and the stop is long enough, we say motor was off
                    return (share < p.get("off_threshold", 0.30)) and long_stop

                # ============================================================
                # Grid helpers for time disaggregation
                # ============================================================

                def first_link_after(tour, post_ts):
                    if post_ts is None: return None, None
                    target = pd.to_datetime(post_ts).timestamp()
                    for lid, ltime in tour:
                        if float(ltime) >= target:
                            hhmm = convert_seconds_to_timestamp(ltime).strftime("%H:%M")
                            h, m = map(int, hhmm.split(":"))
                            return lid, f"{h:02}:{(m // 15) * 15:02}"
                    return None, None

                def add_idle_pollutants(veh_size, link_id, interval, nox_g, hc_g, co_g, emissions_dict):
                    cell = emissions_dict[veh_size][link_id][interval]
                    cell["NOx_idle"] += float(nox_g)
                    cell["HC_idle"]  += float(hc_g)
                    cell["CO_idle"]  += float(co_g)

                def distribute_idle_to_bins(start_ts, dwell_sec, link_id, veh_size, emissions_dict, g_per_sec_effective,
                                            pol_rates_gps=None):
                    """
                    Distribute idle grams into 15-minute cells.
                    Also distributes idle pollutants if pol_rates_gps provided dict with keys nox, hc, co in g/s.
                    """
                    if start_ts is None or dwell_sec <= 0:
                        return
                    t = pd.to_datetime(start_ts).floor("min")
                    remaining = float(dwell_sec)
                    while remaining > 0:
                        step = min(60.0, remaining)
                        hhmm = t.strftime("%H:%M")
                        h, m = map(int, hhmm.split(":"))
                        interval = f"{h:02}:{(m // 15) * 15:02}"
                        add_emission(veh_size, link_id, interval, "idle", g_per_sec_effective * step, emissions_dict)
                        if pol_rates_gps is not None and IDLE_POLLUTANTS_ON:
                            add_idle_pollutants(
                                veh_size, link_id, interval,
                                nox_g=pol_rates_gps["nox"] * step,
                                hc_g=pol_rates_gps["hc"]  * step,
                                co_g=pol_rates_gps["co"]  * step,
                                emissions_dict=emissions_dict
                            )
                        t += pd.to_timedelta(60, unit="s")
                        remaining -= step

                def find_next_positive_load(vehId, start_time_ts, vehicle_status_df):
                    current_time = pd.Timestamp(start_time_ts) + pd.Timedelta(minutes=1)
                    while current_time.strftime("%H:%M") in vehicle_status_df.columns:
                        load_value = vehicle_status_df.loc[vehId, current_time.strftime("%H:%M")]
                        if load_value != -1:
                            return max(0.0, float(load_value))
                        current_time += pd.Timedelta(minutes=1)
                    return 0.0

                def _empty_bucket():
                    return {
                        "drive": 0.0, "idle": 0.0, "cold": 0.0, "total": 0.0,
                        "NOx_idle": 0.0, "HC_idle": 0.0, "CO_idle": 0.0
                    }

                # 1) Aggregation container: totals per base class
                emissions_by_size = {
                    "van [< 2t]": 0.0,
                    "van [> 2t]": 0.0,
                    "truck [10-20 t] + trailer": 0.0,
                    "truck [< 10t]": 0.0,
                }
                total_emissions = 0.0

                # 2) Time grid and link ids MUST be defined BEFORE building emissions_dict
                time_intervals = [f"{h:02}:{m:02}" for h in range(24) for m in [0, 15, 30, 45]]
                link_ids = network_volumes_filtered["link_id"].unique()

                # 3) Build the emissions grid with _empty_bucket() cells
                def init_emissions_grid(emissions_by_size_keys, link_ids, time_intervals):
                    grid = {}
                    for size in emissions_by_size_keys:
                        grid[size] = {}
                        for lid in link_ids:
                            grid[size][lid] = {}
                            for ts in time_intervals:
                                grid[size][lid][ts] = _empty_bucket()
                    return grid

                emissions_dict = init_emissions_grid(emissions_by_size.keys(), link_ids, time_intervals)

                # 4) Robust add_emission for unseen keys
                def add_emission(veh_size, link_id, interval, kind, grams, emissions_dict):
                    if kind not in ("drive", "idle", "cold"):
                        raise ValueError(f"invalid kind {kind}")
                    if veh_size not in emissions_dict:
                        emissions_dict[veh_size] = {}
                    if link_id not in emissions_dict[veh_size]:
                        emissions_dict[veh_size][link_id] = {ts: _empty_bucket() for ts in time_intervals}
                    if interval not in emissions_dict[veh_size][link_id]:
                        emissions_dict[veh_size][link_id][interval] = _empty_bucket()
                    cell = emissions_dict[veh_size][link_id][interval]
                    cell[kind] += float(grams)
                    cell["total"] += float(grams)

                # 5) Per vehicle totals and pollutant tallies
                vehicle_emissions_total = {}
                vehicle_emissions_drive = {}
                vehicle_emissions_idle  = {}
                vehicle_emissions_cold  = {}

                veh_idle_NOx = {}
                veh_idle_HC  = {}
                veh_idle_CO  = {}

                # ============================================================
                # Main loop
                # ============================================================
                
                print(f"[{datetime.datetime.now()}] [POST] Run Main Emissions Loop")   

                n_total = len(vehicle_tour)
                for i, (vehId, tour) in enumerate(vehicle_tour.items()):
                    # ensure presence in result before using maps
                    vehicle_entry = result[result["vehicle_id"] == vehId]
                    if vehicle_entry.empty:
                        if i % 100 == 0 or i == n_total - 1:
                            print(f"[{i:>5}/{n_total}] Vehicle {vehId} not found in result. Skipping.")
                        continue

                    evflag = int(IS_EV_MAP.get(vehId, 0))
                    is_ev = bool(evflag)
                    # classes from maps
                    veh_calc_class = _effective_class_for_calc_from_map(vehId)
                    veh_bucket_class = _base_bucket_class_from_map(vehId)
                    assert veh_calc_class is not None and veh_bucket_class is not None, \
                        f"[EV] Vehicle {vehId} has no valid class mapping (calc={veh_calc_class}, bucket={veh_bucket_class})"

                    # optional informative logging
                    if i % 100 == 0 or i == n_total - 1:
                        prov = _extract_provider_from_vehicle_id(vehId)

                        print(
                            f"[{i:>5}/{n_total}] Vehicle {vehId} | provider={prov} | EV={evflag} | "
                            f"calc_class={veh_calc_class} | bucket_class={veh_bucket_class}"
                        )

                    # optional supply truck filter based on bucket class
                    if WITHOUT_SUPPLY_TRUCKS and veh_bucket_class in SUPPLY_TRUCK_CLASSES:
                        continue

                    drive_emissions_this_vehicle = 0.0
                    idle_emissions_this_vehicle  = 0.0
                    coldstart_emissions_this_vehicle = 0.0

                    idle_NOx_sum = 0.0
                    idle_HC_sum  = 0.0
                    idle_CO_sum  = 0.0

                    # --------------------
                    # Drive links
                    # --------------------
                    for linkId, linkTime in tour:
                        link_ts = convert_seconds_to_timestamp(linkTime)
                        hhmm = link_ts.strftime("%H:%M")
                        hour, minute = map(int, hhmm.split(":"))
                        interval = f"{hour:02}:{(minute // 15) * 15:02}"

                        linkLength = link_length.get(linkId, None)
                        linkType   = link_type.get(linkId, None)
                        if linkLength is None or linkType is None:
                            continue

                        current_load = find_next_positive_load(vehId, link_ts, vehicle_status_df)

                        link_g = calc_emissions_drive(
                            distance_m=linkLength,
                            vehicle_type=veh_calc_class,   # EV aware class for factors
                            segment_type=linkType,
                            load_percentage=current_load,
                            basis=EMISSIONS_BASIS
                        )

                        drive_emissions_this_vehicle += link_g
                        add_emission(veh_bucket_class, linkId, interval, "drive", link_g, emissions_dict)

                    # --------------------
                    # Services for idle and cold
                    # --------------------
                    try:
                        services = vehicle_entry["services"].iloc[0]
                    except Exception:
                        services = []

                    for svc in services:
                        try:
                            dwell_sec = float(svc.getDuration())
                        except Exception:
                            dwell_sec = 0.0
                        if dwell_sec <= 0:
                            continue

                        stop_link = svc.getServiceLink()
                        stop_link_type = link_type.get(stop_link, "urban")
                        curbside = True if stop_link_type == "urban" else False

                        # Idle emissions. EV classes return zero by construction inside your EV handling downstream.
                        idle_g = idle_emissions(
                            dwell_sec=dwell_sec,
                            vehicle_type=veh_calc_class,   # EV aware class for climate gases
                            ambient_temp_c=AMBIENT_TEMP_C,
                            curbside=curbside,
                            t_long_override=T_LONG_OVERRIDE,
                            basis=EMISSIONS_BASIS,
                            is_ev=is_ev
                        )
                        idle_emissions_this_vehicle += idle_g

                        # Distribute idle grams and pollutants if timestamp available
                        start_ts = svc.getAttribute("start_ts", None)
                        if start_ts is not None:
                            start_ts = pd.to_datetime(start_ts)

                            if veh_calc_class in EV_CLASSES:
                                # EV idle tailpipe is zero, no local pollutants
                                g_per_sec_eff = 0.0
                                pol_rates = None
                            else:
                                p = VEH_CLASS_PARAMS.get(veh_bucket_class, DEFAULT_PARAMS)
                                share = engine_on_share(
                                    dwell_sec=dwell_sec,
                                    vehicle_type=veh_bucket_class,   # thresholds by base class
                                    ambient_temp_c=AMBIENT_TEMP_C,
                                    curbside=curbside,
                                    t_long_override=T_LONG_OVERRIDE
                                )
                                # Idle climate gases are TTW only here
                                g_per_sec_eff = p["idle_g_per_sec_ttw"] * share * SEASON_MULT

                                pol_rates = None
                                if IDLE_POLLUTANTS_ON:
                                    mult = IDLE_MULT_BY_CLASS.get(veh_bucket_class, 1.0)
                                    tmult = idle_pollutant_temp_mult(AMBIENT_TEMP_C)
                                    pol_rates = {
                                        "nox": BASE_IDLE_NOX_gps * mult * tmult * share,
                                        "hc":  BASE_IDLE_HC_gps  * mult * tmult * share,
                                        "co":  BASE_IDLE_CO_gps  * mult * tmult * share,
                                    }
                                    idle_NOx_sum += pol_rates["nox"] * dwell_sec
                                    idle_HC_sum  += pol_rates["hc"]  * dwell_sec
                                    idle_CO_sum  += pol_rates["co"]  * dwell_sec

                            distribute_idle_to_bins(
                                start_ts=start_ts,
                                dwell_sec=dwell_sec,
                                link_id=stop_link,
                                veh_size=veh_bucket_class,           # keep base key for grid
                                emissions_dict=emissions_dict,
                                g_per_sec_effective=g_per_sec_eff,
                                pol_rates_gps=pol_rates
                            )

                        # Cold start assignment only for ICE base class being off long enough
                        svc.setAttribute("motor_off", motor_was_off(
                            dwell_sec=dwell_sec,
                            vehicle_type=veh_bucket_class,          # thresholds on base class
                            ambient_temp_c=AMBIENT_TEMP_C,
                            curbside=curbside,
                            t_long_override=T_LONG_OVERRIDE,
                            is_ev=is_ev
                        ))
                        dwell_threshold = effective_cold_dwell_threshold(veh_bucket_class, AMBIENT_TEMP_C)

                        if veh_calc_class not in EV_CLASSES and svc.getAttribute("motor_off", False) and dwell_sec >= dwell_threshold:
                            post_start_ts = svc.getAttribute("post_start_ts", None)
                            link_for_cold, interval_for_cold = first_link_after(tour, post_start_ts)
                            if link_for_cold is None:
                                end_ts = svc.getAttribute("end_ts", None) or (start_ts + pd.to_timedelta(dwell_sec, unit="s"))
                                hhmm = pd.to_datetime(end_ts).strftime("%H:%M")
                                h, m = map(int, hhmm.split(":"))
                                interval_for_cold = f"{h:02}:{(m // 15) * 15:02}"
                                link_for_cold = stop_link

                            cold_em = cold_start_emissions(veh_calc_class, AMBIENT_TEMP_C, basis=EMISSIONS_BASIS)
                            add_emission(veh_bucket_class, link_for_cold, interval_for_cold, "cold", cold_em, emissions_dict)
                            coldstart_emissions_this_vehicle += cold_em

                    # totals and tallies
                    emissions_total_vehicle = drive_emissions_this_vehicle + idle_emissions_this_vehicle + coldstart_emissions_this_vehicle
                    vehicle_emissions_drive[vehId] = drive_emissions_this_vehicle
                    vehicle_emissions_idle[vehId]  = idle_emissions_this_vehicle
                    vehicle_emissions_cold[vehId]  = coldstart_emissions_this_vehicle
                    vehicle_emissions_total[vehId] = emissions_total_vehicle

                    veh_idle_NOx[vehId] = idle_NOx_sum
                    veh_idle_HC[vehId]  = idle_HC_sum
                    veh_idle_CO[vehId]  = idle_CO_sum

                    emissions_by_size[veh_bucket_class] += emissions_total_vehicle
                    total_emissions += emissions_total_vehicle
                    

                # ============================================================
                # Analysis helpers
                # ============================================================
                KINDS = ("drive", "idle", "cold", "total")

                def build_wide_from_emissions(emissions_dict, link_ids, time_intervals, kind="total"):
                    """
                    Returns wide DF: rows = link_id, columns = time_intervals for selected kind.
                    kind in {"drive","idle","cold","total"}.
                    """
                    if kind not in KINDS:
                        raise ValueError(f"Unknown kind {kind}")
                    wide = pd.DataFrame(0.0, index=link_ids, columns=time_intervals)
                    for _, links in (emissions_dict or {}).items():
                        tmp = {}
                        for lid, intervals in (links or {}).items():
                            row = {ts: float((payload or {}).get(kind, 0.0)) for ts, payload in (intervals or {}).items()}
                            if row: tmp[lid] = row
                        if not tmp: continue
                        df_size = pd.DataFrame.from_dict(tmp, orient='index').fillna(0.0)
                        df_size = df_size.reindex(columns=time_intervals, fill_value=0.0)
                        df_size = df_size.reindex(index=link_ids, fill_value=0.0)
                        wide = wide.add(df_size, fill_value=0.0)
                    return wide.reset_index().rename(columns={'index': 'link_id'})

                def build_wide_idle_pollutant(emissions_dict, link_ids, time_intervals, pollutant_key="NOx_idle"):
                    """
                    Aggregate an idle pollutant stored in cells under pollutant_key.
                    """
                    wide = pd.DataFrame(0.0, index=link_ids, columns=time_intervals)
                    for _, links in (emissions_dict or {}).items():
                        tmp = {}
                        for lid, intervals in (links or {}).items():
                            row = {ts: float((payload or {}).get(pollutant_key, 0.0)) for ts, payload in (intervals or {}).items()}
                            if row: tmp[lid] = row
                        if not tmp: continue
                        df_size = pd.DataFrame.from_dict(tmp, orient='index').fillna(0.0)
                        df_size = df_size.reindex(columns=time_intervals, fill_value=0.0)
                        df_size = df_size.reindex(index=link_ids, fill_value=0.0)
                        wide = wide.add(df_size, fill_value=0.0)
                    return wide.reset_index().rename(columns={'index': 'link_id'})

                def to_long(df_wide, time_intervals, id_col='link_id',
                            interval_name='interval_15min', value_name='emissions_g'):
                    value_cols = [c for c in df_wide.columns if c in time_intervals]
                    long_df = df_wide.melt(id_vars=[id_col], value_vars=value_cols,
                                        var_name=interval_name, value_name=value_name)
                    long_df['emissions_kg'] = long_df[value_name] / 1000.0
                    long_df['emissions_t']  = long_df[value_name] / 1_000_000.0
                    return long_df

                def network_totals(long_df, interval_col='interval_15min', value_col='emissions_g'):
                    net = (long_df.groupby(interval_col, as_index=False)[value_col]
                                .sum()
                                .rename(columns={value_col: 'network_emissions_g'}))
                    net['network_emissions_kg'] = net['network_emissions_g'] / 1000.0
                    net['network_emissions_t']  = net['network_emissions_g'] / 1_000_000.0
                    return net

                # Build wide for CO2e by kind
                emissions_df_total = build_wide_from_emissions(emissions_dict, link_ids, time_intervals, kind="total")
                emissions_df_drive = build_wide_from_emissions(emissions_dict, link_ids, time_intervals, kind="drive")
                emissions_df_idle  = build_wide_from_emissions(emissions_dict, link_ids, time_intervals, kind="idle")
                emissions_df_cold  = build_wide_from_emissions(emissions_dict, link_ids, time_intervals, kind="cold")

                # Build wide for idle pollutants
                idle_NOx_wide = build_wide_idle_pollutant(emissions_dict, link_ids, time_intervals, pollutant_key="NOx_idle")
                idle_HC_wide  = build_wide_idle_pollutant(emissions_dict, link_ids, time_intervals, pollutant_key="HC_idle")
                idle_CO_wide  = build_wide_idle_pollutant(emissions_dict, link_ids, time_intervals, pollutant_key="CO_idle")

                # Long forms if needed
                emissions_15min_long = to_long(emissions_df_total, time_intervals)
                idle_NOx_long = to_long(idle_NOx_wide, time_intervals, value_name="NOx_g")
                idle_HC_long  = to_long(idle_HC_wide,  time_intervals, value_name="HC_g")
                idle_CO_long  = to_long(idle_CO_wide,  time_intervals, value_name="CO_g")

                # Example totals
                net_total = network_totals(emissions_15min_long, value_col="emissions_g")
                net_NOx   = network_totals(idle_NOx_long.rename(columns={"NOx_g":"emissions_g"}))
                net_HC    = network_totals(idle_HC_long.rename(columns={"HC_g":"emissions_g"}))
                net_CO    = network_totals(idle_CO_long.rename(columns={"CO_g":"emissions_g"}))

                print("\nSelected checks")
                print(f"CO2e total selected kind total: {emissions_15min_long['emissions_g'].sum()/1000:.2f} kg")
                print(f"Idle NOx total: {idle_NOx_long['NOx_g'].sum():.1f} g, Idle HC total: {idle_HC_long['HC_g'].sum():.1f} g, Idle CO total: {idle_CO_long['CO_g'].sum():.1f} g")

                # ------------------------------------------------------------
                # Add nested 'emissions' column (dict per vehicle)
                # result['emissions'] will hold: {'drive':..., 'idle':..., 'cold':..., 'total':..., 'shares':{...}}
                # ------------------------------------------------------------
                all_vids = set(result['vehicle_id'].tolist()) | set(vehicle_emissions_total.keys())

                vehicle_emissions_dict = {}
                for vid in all_vids:
                    d = float(vehicle_emissions_drive.get(vid, 0.0) or 0.0)
                    i = float(vehicle_emissions_idle.get(vid, 0.0) or 0.0)
                    c = float(vehicle_emissions_cold.get(vid, 0.0) or 0.0)
                    t = float(vehicle_emissions_total.get(vid, 0.0) or 0.0)
                    if t > 0:
                        idle_sh = i / t
                        cold_sh = c / t
                    else:
                        idle_sh = 0.0
                        cold_sh = 0.0

                    vehicle_emissions_dict[vid] = {
                        "drive": d,
                        "idle": i,
                        "cold": c,
                        "total": t,
                        "shares": {
                            "idle": idle_sh,
                            "cold": cold_sh
                        }
                    }

                # Map back to the result dataframe
                result["emissions"] = result["vehicle_id"].map(vehicle_emissions_dict)


                # Add name column (robust to ints/strings in main_area_type)
                result["main_area_type_name"] = (
                    result["main_area_type"].astype(str).map(category_dict).fillna("Unknown")
                )

                # Annahmen:
                # - emissions_15min_long hat Spalten: link_id, interval_15min, emissions_g
                # - link_raumtyp: dict {link_id: area_type_id_oder_name}
                # Optional:
                # - area_type_label: dict {area_type_id: "Metropolitan Center", ...} falls du IDs auf Namen mappen willst

                # 1) Area Type an die 15-Minuten-Daten hängen
                emissions_15min_long['area_type_id'] = emissions_15min_long['link_id'].map(link_raumtyp)

                # Optional lesbarer Name
                if 'area_type_label' in globals() and isinstance(area_type_label, dict):
                    emissions_15min_long['area_type'] = emissions_15min_long['area_type_id'].map(area_type_label)
                else:
                    # Falls keine Label Map vorliegt: nimm die ID als Name, ersetze fehlende durch "Unknown"
                    emissions_15min_long['area_type'] = emissions_15min_long['area_type_id'].fillna('Unknown')

                # 2) Merge mit Netzgeometrie beibehalten, jetzt inklusive area_type
                merged_15min = pd.merge(
                    network_volumes_filtered,
                    emissions_15min_long,
                    on='link_id',
                    how='left'
                )

                # 3) Netzsummen je 15-Minuten-Intervall UND Raumtyp
                network_15min_by_area = (
                    emissions_15min_long
                    .groupby(['interval_15min', 'area_type'], as_index=False)['emissions_g']
                    .sum()
                    .rename(columns={'emissions_g': 'emissions_g_sum'})
                )
                network_15min_by_area['emissions_kg_sum'] = network_15min_by_area['emissions_g_sum'] / 1000.0

                # 4) Optional: Pivot für einfache Plots oder Export
                network_15min_pivot = network_15min_by_area.pivot(
                    index='interval_15min',
                    columns='area_type',
                    values='emissions_kg_sum'
                ).reindex(index=time_intervals).fillna(0.0)

                # 5) Reporting Beispiele
                print("\nBeispiel je Link, Intervall und Area Type:")
                print(emissions_15min_long[['link_id', 'interval_15min', 'area_type', 'emissions_g']].head())

                print("\nNetzweite Summen je 15-Minuten-Intervall und Area Type:")
                print(network_15min_by_area.head())

                print("\nPivot je Intervall und Area Type in kg:")
                print(network_15min_pivot.head())  
                
                # ===== Persist all computed results =====
                save_targets = {
                    # Main enriched per vehicle result
                    "emissions_result": result,

                    # Gridded emissions store
                    "emissions_grid": emissions_dict,

                    # Wide CO2e by kind
                    "emissions_wide_total": emissions_df_total,
                    "emissions_wide_drive": emissions_df_drive,
                    "emissions_wide_idle":  emissions_df_idle,
                    "emissions_wide_cold":  emissions_df_cold,

                    # Wide idle pollutants
                    "idle_NOx_wide": idle_NOx_wide,
                    "idle_HC_wide":  idle_HC_wide,
                    "idle_CO_wide":  idle_CO_wide,

                    # Long forms
                    "emissions_15min_long": emissions_15min_long,
                    "idle_NOx_long": idle_NOx_long,
                    "idle_HC_long":  idle_HC_long,
                    "idle_CO_long":  idle_CO_long,

                    # Network totals by 15 min
                    "network_totals_total": net_total,
                    "network_totals_NOx":   net_NOx,
                    "network_totals_HC":    net_HC,
                    "network_totals_CO":    net_CO,

                    # Network totals by area type
                    "network_15min_by_area": network_15min_by_area,
                    "network_15min_pivot":   network_15min_pivot,

                    # Per vehicle tallies
                    "vehicle_emissions_total": vehicle_emissions_total,
                    "vehicle_emissions_drive": vehicle_emissions_drive,
                    "vehicle_emissions_idle":  vehicle_emissions_idle,
                    "vehicle_emissions_cold":  vehicle_emissions_cold,
                    "vehicle_emissions_dict":  vehicle_emissions_dict,

                    # Optional helpers if you want them persisted as well
                    "veh_idle_NOx": veh_idle_NOx,
                    "veh_idle_HC":  veh_idle_HC,
                    "veh_idle_CO":  veh_idle_CO,
                }

                # Write each object as <run_name>_<key>.pkl under OUTPUT_BASE
                for key, obj in save_targets.items():
                    try:
                        _safe_save_obj(obj, os.path.join(OUTPUT_BASE, f"{r['run_name']}_{key}"))
                    except Exception as e:
                        print(f"[WARN] Could not save {key}: {e}")

                print("All result bundles written to:", OUTPUT_BASE)    
                print(f"[{datetime.datetime.now()}] [INFO] Script execution completed successfully. (Total Runtime: {format_runtime(time.time() - overall_start_time)})")   


            except Exception as e:
                print('[WARN] run_emissions_pipeline failed:', e)                       

        finally:
            progress.update(i, prefix=f"Processing {r['scenario']}/{r['run_name']}")

    summary = pd.DataFrame(rows)
    try:
        summary.to_parquet(os.path.join(OUTPUT_BASE, 'summary_impact.parquet'), index=False)
    except Exception:
        summary.to_pickle(os.path.join(OUTPUT_BASE, 'summary_impact.pkl'))
    print('Impact batch complete. Summary written to', os.path.join(OUTPUT_BASE, 'summary_impact.*'))
    


<>:11: SyntaxWarning: invalid escape sequence '\e'
<>:11: SyntaxWarning: invalid escape sequence '\e'
C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_26160\3057880396.py:11: SyntaxWarning: invalid escape sequence '\e'
  OUTPUT_BASE = os.path.join(BATCH_DIR, "processed\emissions_fix")  # write alongside matsim processed


In [8]:
run_batch()

Discovered 24 runs across scenarios: ['basecase', 'batchhigh', 'batchmedium', 'batchmoderate']
Processing basecase/basecase_12052025_iter150_jsprit100 [------------------------------] 0/24 | elapsed    0.0s | ETA    0.0s[2025-10-09 20:04:12.192908] [1/10] Initiating script execution...
[2025-10-09 20:04:12.192937] [2/10] Loading events from C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\basecase Week\basecase_12052025_iter150_jsprit100\basecase_12052025.output_events.xml.gz...
[2025-10-09 20:04:21.841498] [INFO] Loaded events for 1913 vehicles. (Runtime: 9.65 s)
[2025-10-09 20:04:21.841702] [INFO] Number of parsed events: 1307935
[2025-10-09 20:04:21.841715] [INFO] Network size: 544514
[2025-10-09 20:04:21.841730] [3/10] Processing network volumes...
[2025-10-09 20:04:36.610248] [INFO] Network volumes processed. (Runtime: 14.77 s)
[2025-10-09 20:04:36.610699] [INFO] Reduced Network size to: 77916 (85.69% reduction)
[2025-10-09 20:04:36.610715] [4/10] Ca

C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_26160\1641712650.py:613: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  plot_data["Start time"][df_start.iloc[event]["person"]] = df_start.iloc[event]['time']
C:\Users\bienzeisler\AppData\Loca

[2025-10-09 20:05:08.231792] [INFO] Plot data created. (Runtime: 29.53 s)
[2025-10-09 20:05:08.232079] [8/10] Integrating plot data with vehicle data...
[2025-10-09 20:05:08.235834] [INFO] Data integrated. (Runtime: 3.74 ms)
[2025-10-09 20:05:08.235873] [9/10] Parsing carriers from XML at C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\basecase Week\basecase_12052025_iter150_jsprit100\basecase_12052025.output_carriers.xml.gz...
[2025-10-09 20:05:17.585243] [INFO] Carriers parsed. (Runtime: 9.35 s)
[2025-10-09 20:05:17.585500] [10/10] Augmenting result data with vehicle demand information...

📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1651
✅ Found: 1651
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
[2025-10-09 20:08:48.493783] [POST] Attaching service timestamps from events to Service objects...
[2025-10-09 20:09:05.852671] [INFO] Service timestamps attached. (Runtime: 17.36 s)
[2025-10-09 20:09:05.85

Processing vehicle tours: 100%|██████████| 1651/1651 [01:00<00:00, 27.50tour/s]


[2025-10-09 20:10:05.924010] [POST] Filter blow trashhold:  0.05
[INFO] result not present; skipping low-util filter.
[2025-10-09 20:10:05.924604] [POST] Run EV Model

[EV] Sanity report
Global target: 347 EVs (21.0% of 1651 vehicles)
Assigned EVs: 347 (21.0%)

amazon     | Vehicles:  302 | Vans:  302 | EV target:   34 ( 11.3%) | EV assigned:   34 ( 11.3%)
dhl        | Vehicles:  569 | Vans:  569 | EV target:  274 ( 48.2%) | EV assigned:  274 ( 48.2%)
dpd        | Vehicles:  164 | Vans:  164 | EV target:    6 (  3.7%) | EV assigned:    6 (  3.7%)
fedex      | Vehicles:  124 | Vans:  124 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
gls        | Vehicles:  148 | Vans:  148 | EV target:    6 (  4.1%) | EV assigned:    6 (  4.1%)
hermes     | Vehicles:  192 | Vans:  192 | EV target:   21 ( 10.9%) | EV assigned:   21 ( 10.9%)
ups        | Vehicles:  152 | Vans:  152 | EV target:    6 (  3.9%) | EV assigned:    6 (  3.9%)
TOTAL     | Vehicles: 1651 | Target:  347 (21.0%) | Assigne

...

[2025-10-10 00:57:23.018090] [POST] Filter blow trashhold:  0.05
[INFO] result not present; skipping low-util filter.
[2025-10-10 00:57:23.018717] [POST] Run EV Model

[EV] Sanity report
Global target: 329 EVs (21.0% of 1565 vehicles)
Assigned EVs: 329 (21.0%)

amazon     | Vehicles:  245 | Vans:  245 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
dhl        | Vehicles:  692 | Vans:  692 | EV target:  305 ( 44.1%) | EV assigned:  305 ( 44.1%)
dpd        | Vehicles:  242 | Vans:  242 | EV target:    7 (  2.9%) | EV assigned:    7 (  2.9%)
fedex      | Vehicles:   46 | Vans:   46 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
gls        | Vehicles:  115 | Vans:  115 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
hermes     | Vehicles:  164 | Vans:  164 | EV target:   17 ( 10.4%) | EV assigned:   17 ( 10.4%)
ups        | Vehicles:   61 | Vans:   61 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
TOTAL     | Vehicles: 1565 | Target:  329 (21.0%) | Assigne

C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_26160\1641712650.py:613: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  plot_data["Start time"][df_start.iloc[event]["person"]] = df_start.iloc[event]['time']
C:\Users\bienzeisler\AppData\Loca

[2025-10-10 01:04:58.777995] [INFO] Plot data created. (Runtime: 23.28 s)
[2025-10-10 01:04:58.778220] [8/10] Integrating plot data with vehicle data...
[2025-10-10 01:04:58.781561] [INFO] Data integrated. (Runtime: 3.33 ms)
[2025-10-10 01:04:58.781597] [9/10] Parsing carriers from XML at C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\batch\batchmoderate Week\batchmoderate_17052025_iter150_jsprit100\batchmoderate_17052025.output_carriers.xml.gz...
[2025-10-10 01:05:08.917790] [INFO] Carriers parsed. (Runtime: 10.14 s)
[2025-10-10 01:05:08.918092] [10/10] Augmenting result data with vehicle demand information...

📋 Vehicle ID Validation Report
──────────────────────────────
🔢 Expected IDs total: 1145
✅ Found: 1145
❌ Missing: 0
🔁 Duplicates: 0
⚠️ Empty vehicle_id entries: 0
[2025-10-10 01:06:55.250860] [POST] Attaching service timestamps from events to Service objects...
[2025-10-10 01:07:05.024095] [INFO] Service timestamps attached. (Runtime: 9.77 s)
[2025-10

Processing vehicle tours: 100%|██████████| 1145/1145 [00:45<00:00, 25.21tour/s]


[2025-10-10 01:07:50.475912] [POST] Filter blow trashhold:  0.05
[INFO] result not present; skipping low-util filter.
[2025-10-10 01:07:50.476281] [POST] Run EV Model

[EV] Sanity report
Global target: 240 EVs (21.0% of 1145 vehicles)
Assigned EVs: 240 (21.0%)

amazon     | Vehicles:  272 | Vans:  272 | EV target:   29 ( 10.7%) | EV assigned:   29 ( 10.7%)
dhl        | Vehicles:  368 | Vans:  368 | EV target:  177 ( 48.1%) | EV assigned:  177 ( 48.1%)
dpd        | Vehicles:   90 | Vans:   90 | EV target:    3 (  3.3%) | EV assigned:    3 (  3.3%)
fedex      | Vehicles:   35 | Vans:   35 | EV target:    0 (  0.0%) | EV assigned:    0 (  0.0%)
gls        | Vehicles:   83 | Vans:   83 | EV target:    2 (  2.4%) | EV assigned:    2 (  2.4%)
hermes     | Vehicles:  249 | Vans:  249 | EV target:   28 ( 11.2%) | EV assigned:   28 ( 11.2%)
ups        | Vehicles:   48 | Vans:   48 | EV target:    1 (  2.1%) | EV assigned:    1 (  2.1%)
TOTAL     | Vehicles: 1145 | Target:  240 (21.0%) | Assigne